In [ ]:
# ===== SEL IDENTITAS - JANGAN DIHAPUS, HARUS SEL PERTAMA =====
import sys, platform, hashlib, datetime

try:
    import zoneinfo
    _ZONA = zoneinfo.ZoneInfo('Asia/Jakarta')
except Exception:            # zoneinfo / tzdata tidak tersedia -> jatuh ke UTC, tidak crash
    _ZONA = datetime.timezone.utc

NAMA = 'Singgih Wahyu Permana'
NIM = '244107020102'         # ganti dengan NIM Anda
KELAS = 'TI 3E'

N = int(NIM[-3:])
PARAM = {
    'bins': 2 ** ((N % 4) + 3),
    'clip': round(1.0 + (N % 5) * 0.5, 1),
    'tile': (N % 4) + 2,
    'L': 2 ** ((N % 3) + 1),
}
waktu = datetime.datetime.now(_ZONA)

print('NAMA   :', NAMA)
print('NIM    :', NIM, '| N =', N)
print('KELAS  :', KELAS)
for k, v in PARAM.items():
    print('%-6s :' % k, v)
print('WAKTU  :', waktu.strftime('%Y-%m-%d %H:%M:%S %Z'))
print('SISTEM :', sys.version.split()[0], platform.platform())
kunci = NIM + '|' + waktu.strftime('%Y-%m-%d')
print('TOKEN  :', hashlib.sha256(kunci.encode()).hexdigest()[:16])

# Modul 5 — Histogram, Histogram Equalization, dan Dithering

**Pengolahan Citra dan Visi Komputer — Jurusan Teknologi Informasi**
**Singgih Wahyu Permana — 244107020102 — TI 3E**

Notebook ini adalah pengerjaan **lengkap** Modul 5 pada lingkungan **Kaggle** (tanpa
`google.colab`, tanpa `drive.mount`, tanpa `files.upload()`, tanpa GUI). Seluruh berkas citra
diakses melalui variabel `BASE`/`WEEK` yang otomatis menunjuk ke
`/kaggle/input/pcvk-images/week5` (Kaggle) atau folder `datasets/week5` (lokal).

Parameter percobaan diturunkan dari NIM: N = **102**, sehingga
`bins = 32`, `clip = 2.0`, `tile = 4`, `L = 2`.

Cakupan notebook:

| Bagian | Isi |
|---|---|
| Sel identitas | Sel pertama wajib: NAMA, NIM, PARAM, TOKEN (zona `Asia/Jakarta`) |
| E-1 | Histogram manual, `np.histogram`, `cv.calcHist` + bukti selisih nol; p(j) & CDF 2x2; bins 32 vs 256 |
| E-2 | Histogram equalization manual vs `cv.equalizeHist`, penyebab selisih, PSNR; ekualisasi warna cara 1 vs cara 2 (YCrCb/HSV/LAB) + ukur pergeseran Hue; CLAHE `clipLimit=2.0`, `tileGridSize=(4,4)` |
| E-3 | Dithering Floyd-Steinberg manual (L=2), verifikasi satu langkah manual, HE-lalu-dither vs dither langsung, dithering citra berwarna 3 kanal |
| PERTANYAAN | E1 no.1-3 dan E2 no.1-2 dijawab pada sel markdown dengan angka keluaran program |

Setiap citra keluaran diberi judul yang memuat NIM (`T()` = `NIM + ' - ' + teks`).
Semua fungsi yang diminta "manual" (`histogram_manual`, `he_manual`, `floyd_steinberg`) tidak
memanggil fungsi library yang bersangkutan (`np.histogram`/`cv.calcHist`, `cv.equalizeHist`,
`cv.medianBlur`, dsb.).

In [ ]:
# ==== KONFIGURASI LINGKUNGAN (jalan di Kaggle maupun lokal) ====
import os, sys, glob, math
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

def _find_base():
    for p in ('/kaggle/input/pcvk-images',
              '/kaggle/input/datasets/singgihwahyupermana/pcvk-images'):
        if os.path.isdir(p):
            return p
    _hit = glob.glob('/kaggle/input/**/pcvk-images', recursive=True)
    if _hit:
        return _hit[0]
    d = os.getcwd()
    for _ in range(6):
        cand = os.path.join(d, 'datasets')
        if os.path.isdir(cand):
            return cand
        d = os.path.dirname(d)
    return None

BASE = _find_base()
WEEK = os.path.join(BASE, 'week5')   # <-- ganti angka minggu sesuai notebook

def show(img, title='', cmap=None, figsize=(6, 4)):
    plt.figure(figsize=figsize)
    if img.ndim == 3:
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.show()

# Shim cv2_imshow (Colab) -> Kaggle
try:
    from google.colab.patches import cv2_imshow
except Exception:
    def cv2_imshow(img):
        show(img)

print('BASE :', BASE)
print('WEEK :', WEEK)
print('files:', sorted(os.listdir(WEEK)) if WEEK and os.path.isdir(WEEK) else 'MISSING')

In [ ]:
# ---- Pemetaan istilah modul -> struktur folder Kaggle/lokal ----
# Modul asli memakai CONTOH (citra contoh bersama) dan BASE (folder citra KTM pribadi).
# Di Kaggle/lokal keduanya berada pada satu folder minggu: WEEK.
CONTOH = WEEK     # lena.jpg, lena_lc.jpg
KTMDIR = WEEK     # KTM1a.jpg ... KTM1d.jpg
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

def p(nama):
    """path lengkap sebuah berkas citra pada folder minggu ini"""
    return os.path.join(WEEK, nama)

def T(teks):
    """semua judul citra pada Week5 wajib memuat NIM"""
    return NIM + ' - ' + teks

import time

print('CONTOH :', CONTOH)
print('KTMDIR :', KTMDIR)
print('OUT    :', OUT)
for _n in ['lena.jpg', 'lena_lc.jpg', 'KTM1a.jpg', 'KTM1b.jpg', 'KTM1c.jpg', 'KTM1d.jpg']:
    print('  %-12s %s' % (_n, 'ADA' if os.path.isfile(p(_n)) else 'TIDAK ADA'))

## Catatan Lingkungan (Colab → Kaggle)

Modul asli meminta menghubungkan Google Colab ke GitHub lalu memakai dua folder:

```python
CONTOH = '/content/drive/MyDrive/PCVK/Images'        # citra contoh bersama
BASE   = '/content/drive/MyDrive/PCVK/Images/' + NIM # citra KTM milik Anda
```

Pada versi Kaggle ini tidak ada Drive maupun Colab, sehingga kedua peran itu dipenuhi oleh satu
folder minggu: `WEEK`. Sel konfigurasi menyediakan

| Nama pada modul | Padanan di notebook ini | Isi |
|---|---|---|
| `CONTOH` | `CONTOH = WEEK` | `lena.jpg`, `lena_lc.jpg` (tahap 1 / verifikasi kode) |
| `BASE` | `KTMDIR = WEEK` | `KTM1a.jpg` … `KTM1d.jpg` (tahap 2 / bahan analisis) |
| `show()` | `show(img, title)` | pengganti `cv2_imshow` (shim tetap disediakan) |
| `T(teks)` | `NIM + ' - ' + teks` | semua judul citra wajib memuat NIM |

Setiap percobaan dikerjakan **dua tahap** seperti ketentuan modul: tahap 1 dengan citra contoh
(`lena.jpg` / `lena_lc.jpg`) untuk memastikan kode benar, tahap 2 dengan citra KTM sendiri
(`KTM1a.jpg` … `KTM1d.jpg`) sebagai bahan analisis dan penilaian.

## D.1 Histogram Citra

Histogram citra adalah representasi grafis distribusi intensitas piksel. Sumbu horizontal
menyatakan variasi nilai piksel (0–255 untuk citra 8-bit) dan sumbu vertikal menyatakan jumlah
piksel pada intensitas tersebut. Sisi kiri sumbu horizontal mewakili area gelap, tengah area
abu-abu, dan kanan area terang. Citra grayscale memiliki satu histogram; citra berwarna
memiliki satu histogram untuk setiap kanal B, G, dan R.

Alur percobaan (sesuai flowchart modul):

```mermaid
flowchart TD
    A[Start] --> B[Input image]
    B --> C[Menghitung jumlah kemunculan tiap nilai piksel<br/>pada seluruh piksel citra]
    C --> D[Histogram: sumbu x intensitas,<br/>sumbu y jumlah kemunculan]
    D --> E[Histogram Citra] --> F[End]
```

Algoritma loop bersarang ditulis apa adanya agar alurnya terlihat, lalu diverifikasi terhadap
`np.histogram` dan `cv.calcHist`, dan untuk citra berwarna diulang pada tiap kanal.

### E-1 Langkah 3 (Tahap 1) — histogram manual pada lena.jpg

In [ ]:
# ---- E-1 langkah 3 : histogram citra dibuat manual (loop bersarang) ----
def histogram_manual(img, L=256):
    h = np.zeros(L, dtype=np.int64)      # 1. siapkan penghitung, isi 0
    for y in range(img.shape[0]):        # 2. telusuri setiap baris
        for x in range(img.shape[1]):    #    dan setiap kolom
            h[img[y, x]] += 1            # 3. tambah 1 sesuai nilai piksel
    return h

# ---- Tahap 1 : citra contoh lena.jpg ----
lena_g = cv.imread(p('lena.jpg'), cv.IMREAD_GRAYSCALE)
t0 = time.perf_counter()
h_lena = histogram_manual(lena_g)
t_manual_lena = time.perf_counter() - t0
p_lena = h_lena / lena_g.size            # 4. normalisasi, jumlahnya = 1
cdf_lena = np.cumsum(p_lena)             # 5. distribusi kumulatif
assert h_lena.sum() == lena_g.size       # pemeriksaan wajib

print('Tahap 1 - lena.jpg :', lena_g.shape, '| total piksel', lena_g.size)
print('assert h.sum() == gray.size  -> OK')
print('jumlah p(j) = %.6f | CDF terakhir = %.6f' % (p_lena.sum(), cdf_lena[-1]))
print('puncak histogram : intensitas %d dengan %d piksel (p = %.4f)'
      % (int(h_lena.argmax()), int(h_lena.max()), p_lena[int(h_lena.argmax())]))
print('waktu loop bersarang pada lena.jpg : %.3f s' % t_manual_lena)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(lena_g, cmap='gray'); plt.title(T('lena.jpg - citra asli (grayscale)')); plt.axis('off')
plt.subplot(1, 2, 2)
plt.plot(h_lena, color='tab:blue', lw=0.8)
plt.title(T('lena.jpg - histogram manual 256 bin'))
plt.xlabel('intensitas warna'); plt.ylabel('jumlah kemunculan')
plt.tight_layout(); plt.show()

**Hasil tahap 1 (`lena.jpg`, 512x512 = 262.144 piksel):** pemeriksaan wajib
`h.sum() == gray.size` lolos; `p(j)` berjumlah tepat 1,000000 dan CDF berakhir di 1,000000.
Puncak histogram jatuh pada intensitas **155** dengan **2.707 piksel** (`p = 0,0103`), sesuai
dengan citra uji standar yang terdistribusi cukup merata. Waktu loop bersarang Python
≈ 0,15–0,25 s untuk 262.144 piksel — masih nyaman untuk citra contoh, tetapi akan terasa berat
pada citra KTM yang 2,5 kali lebih besar (lihat tahap 2).

### PERTANYAAN E1 no.1 — pembuktian tiga cara, tahap 1 pada `lena.jpg`

Sesuai permintaan soal, pembuktian dilakukan lebih dulu pada `lena.jpg`, baru
kemudian diulang pada `KTM1b.jpg`.

In [ ]:
# ---- PERTANYAAN E1 no.1 : histogram manual vs numpy.histogram vs cv.calcHist ----
def banding_histogram(gray, nama):
    """hitung histogram tiga cara dan buktikan selisih maksimumnya nol"""
    t0 = time.perf_counter(); h_man = histogram_manual(gray); t_man = time.perf_counter() - t0
    t0 = time.perf_counter()
    h_np, _ = np.histogram(gray, bins=256, range=(0, 256))
    t_np = time.perf_counter() - t0
    t0 = time.perf_counter()
    h_cv = cv.calcHist([gray], [0], None, [256], [0, 256]).ravel()
    t_cv = time.perf_counter() - t0

    d_mn = np.abs(h_man.astype(np.float64) - h_np.astype(np.float64)).max()
    d_mc = np.abs(h_man.astype(np.float64) - h_cv.astype(np.float64)).max()
    d_nc = np.abs(h_np.astype(np.float64) - h_cv.astype(np.float64)).max()

    print('== %s ==' % nama)
    print('   waktu  manual : %8.4f s | np.histogram : %9.6f s | cv.calcHist : %9.6f s'
          % (t_man, t_np, t_cv))
    print('   max |manual - numpy| = %.1f' % d_mn)
    print('   max |manual - cv2  | = %.1f' % d_mc)
    print('   max |numpy  - cv2  | = %.1f' % d_nc)
    assert d_mn == 0 and d_mc == 0 and d_nc == 0, 'hasil ketiga histogram tidak identik'
    print('   -> KETIGA HASIL IDENTIK, selisih maksimum = 0')
    return h_man, h_np, h_cv

banding_histogram(lena_g, 'Tahap 1 : lena.jpg (grayscale)')

### E-1 Langkah 3 (Tahap 2) — histogram manual pada KTM1b.jpg

Percobaan yang sama diulang pada citra KTM sendiri (sekaligus sebagai pembuktian
PERTANYAAN E1 no.1 pada citra KTM):

In [ ]:
# ---- Tahap 2 : ulangi pembuktian pada KTM1b.jpg milik sendiri ----
ktm1b_g = cv.imread(p('KTM1b.jpg'), cv.IMREAD_GRAYSCALE)
h_ktm1b, h_np_b, h_cv_b = banding_histogram(ktm1b_g, 'Tahap 2 : KTM1b.jpg (grayscale)')
print()
print('ukuran KTM1b.jpg :', ktm1b_g.shape, '| total piksel :', ktm1b_g.size)

selisih = np.abs(h_ktm1b.astype(np.float64) - h_cv_b.astype(np.float64))
plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1)
plt.imshow(ktm1b_g, cmap='gray'); plt.title(T('KTM1b.jpg - citra asli')); plt.axis('off')
plt.subplot(1, 3, 2)
plt.plot(h_ktm1b, color='tab:blue', lw=1, label='manual')
plt.plot(h_np_b, color='tab:orange', lw=4, alpha=.4, label='np.histogram')
plt.plot(h_cv_b, '--', color='tab:green', lw=1, label='cv.calcHist')
plt.title(T('KTM1b.jpg - tiga histogram bertumpuk')); plt.legend(fontsize=8)
plt.xlabel('intensitas'); plt.ylabel('jumlah piksel')
plt.subplot(1, 3, 3)
plt.plot(selisih, color='tab:red', lw=1)
plt.title(T('|manual - cv2| per bin (maks = %.0f)' % selisih.max()))
plt.xlabel('intensitas'); plt.ylabel('selisih piksel'); plt.ylim(-1, 1)
plt.tight_layout(); plt.show()

**Hasil tahap 2 (`KTM1b.jpg`, 638x1011 = 645.018 piksel):** semua pemeriksaan lolos dan
histogram manual identik dengan kedua fungsi library. Karena jumlah pikselnya 2,46 kali lipat
lena, waktu loop bersarang naik menjadi ≈ 0,35–0,55 s. Ini menunjukkan biaya `histogram_manual`
tumbuh linier terhadap jumlah piksel (645.018 iterasi interpreter), sedangkan `np.histogram` dan `cv.calcHist` (yang
diimplementasikan di level C dengan vektorisasi/paralelisasi) menyelesaikan pekerjaan yang sama
dalam orde milidetik atau kurang: sekitar 30 kali (`np.histogram`) sampai ratusan kali
(`cv.calcHist`) lebih cepat.

### Histogram citra berwarna (KTM1d.jpg)

In [ ]:
# ---- histogram citra berwarna : ulangi loop manual untuk tiap kanal (KTM1d.jpg) ----
bgr_ktm = cv.imread(p('KTM1d.jpg'))
print('KTM1d.jpg :', bgr_ktm.shape)
h_kanal = [histogram_manual(bgr_ktm[:, :, i]) for i in range(3)]
for i, nama in enumerate(['Biru (B)', 'Hijau (G)', 'Merah (R)']):
    print('kanal %-9s rerata %6.1f | puncak di intensitas %3d'
          % (nama, bgr_ktm[:, :, i].mean(), int(h_kanal[i].argmax())))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv.cvtColor(bgr_ktm, cv.COLOR_BGR2RGB)); plt.title(T('KTM1d.jpg - citra berwarna')); plt.axis('off')
plt.subplot(1, 2, 2)
for i, nama in enumerate(['Biru', 'Hijau', 'Merah']):
    plt.plot(h_kanal[i], label=nama, lw=1)
plt.title(T('KTM1d.jpg - histogram manual tiap kanal B, G, R'))
plt.xlabel('intensitas'); plt.ylabel('jumlah piksel'); plt.legend()
plt.tight_layout(); plt.show()

## Jawaban PERTANYAAN E1 no.1 — manual vs `np.histogram` vs `cv.calcHist`

Program menghitung histogram yang sama dengan tiga cara, lalu menghitung selisih maksimum antar
hasil (elemen demi elemen, seluruh 256 bin):

| Tahap | Berkas | max \|manual − numpy\| | max \|manual − cv2\| | max \|numpy − cv2\| | Kesimpulan |
|---|---|---|---|---|---|
| 1 | `lena.jpg` | **0** | **0** | **0** | ketiga histogram identik |
| 2 | `KTM1b.jpg` | **0** | **0** | **0** | ketiga histogram identik |

Selisih maksimumnya nol, jadi ketiga cara menghasilkan histogram yang **sama persis** — bukan
sekadar mirip. Buktinya ditampilkan dua kali: pada `lena.jpg` (tahap verifikasi) dan pada
`KTM1b.jpg` (citra KTM sendiri), dan grafik selisih per bin digambar rata di nol.

Yang berbeda hanyalah waktu tempuh, dan perbedaannya sangat besar. Pada `KTM1b.jpg` (645.018
piksel) loop bersarang Python memerlukan ≈ 0,36–0,47 s, `np.histogram` ≈ 0,009–0,013 s, dan
`cv.calcHist` ≈ 0,0003–0,0013 s — jadi `np.histogram` sekitar 30–40 kali lebih cepat daripada
loop Python, dan `cv.calcHist` sekitar 300–1000 kali lebih cepat. Pada `lena.jpg` polanya sama
(manual ≈ 0,15 s, `np.histogram` ≈ 0,006 s, `cv.calcHist` ≈ 0,007 s); panggilan `cv.calcHist`
pertama tampak hanya sedikit lebih cepat karena masih menyertakan biaya inisialisasi OpenCV,
sedangkan pada `KTM1b.jpg` — panggilan berikutnya — biaya itu sudah hilang. Penyebabnya: loop bersarang melakukan satu iterasi interpreter per piksel
(645.018 iterasi), sedangkan NumPy memakai operasi vektor di level C, dan `cv.calcHist`
menggunakan rutin OpenCV yang sudah dioptimalkan (SIMD/paralel) sekaligus menghindari pembuatan
array perantara. Kesimpulannya: loop manual benar secara matematis dan wajib ditulis untuk
memahami alur, tetapi untuk pemakaian sehari-hari `cv.calcHist`/`np.histogram` yang dipakai.

In [ ]:
# ---- PERTANYAAN E1 no.2 : histogram ternormalisasi p(j) + kurva CDF dalam 2 x 2 ----
berkas = ['KTM1a.jpg', 'KTM1b.jpg', 'KTM1c.jpg', 'KTM1d.jpg']
kondisi = ['ruangan gelap', 'lampu ruangan', 'flash', 'luar / matahari']
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
stat = []
for ax, nm, kd in zip(axes.ravel(), berkas, kondisi):
    g = cv.imread(p(nm), cv.IMREAD_GRAYSCALE)
    h = np.bincount(g.ravel(), minlength=256).astype(np.float64)
    pj = h / g.size                       # p(j) ternormalisasi
    cdfj = np.cumsum(pj)                  # CDF(j)
    ax.plot(pj, color='tab:blue', lw=0.8)
    ax.set_xlabel('intensitas j'); ax.set_ylabel('p(j)', color='tab:blue')
    ax.tick_params(axis='y', labelcolor='tab:blue')
    ax.set_title(T('{} - {}'.format(nm, kd)), fontsize=9)
    ax2 = ax.twinx()
    ax2.plot(cdfj, color='tab:red', lw=1.8)
    ax2.set_ylabel('CDF(j)', color='tab:red'); ax2.set_ylim(0, 1.05)
    ax2.tick_params(axis='y', labelcolor='tab:red')
    stat.append((nm, kd, g.mean(), np.median(g), np.percentile(g, 5), np.percentile(g, 95),
                 pj[:33].sum() * 100, pj[224:].sum() * 100, g.std(),
                 int((cdfj < 0.5).sum()), int(np.percentile(g, 90))))
plt.tight_layout(); plt.show()

print('%-10s %-15s %7s %7s %5s %5s %9s %9s %6s %6s %6s'
      % ('berkas', 'kondisi', 'rerata', 'median', 'p5', 'p95', '<=32%', '>=224%', 'std', 'CDF50', 'p90'))
for s in stat:
    print('%-10s %-15s %7.1f %7.0f %5.0f %5.0f %9.1f %9.1f %6.1f %6d %6d' % s)

## Jawaban PERTANYAAN E1 no.2 — p(j), CDF, dan kondisi pencahayaan

Gambar 2x2 berikut memuat histogram ternormalisasi $p(j) = n_j / MN$ (garis biru, sumbu kiri) dan
kurva CDF $CDF(j) = \sum_{k \le j} p(k)$ (garis merah, sumbu kanan) untuk keempat citra KTM.

| Berkas | Kondisi akuisisi | Rerata | Median | p5 | p95 | piksel ≤ 32 | piksel ≥ 224 | Std | CDF(50%) | p90 |
|---|---|---|---|---|---|---|---|---|---|---|
| KTM1a.jpg | ruangan gelap | 29,9 | 35 | 6 | 35 | 18,4 % | 0,0 % | 11,4 | 35 | 35 |
| KTM1b.jpg | lampu ruangan | 119,5 | 136 | 46 | 136 | 2,9 % | 0,0 % | 37,4 | 136 | 136 |
| KTM1c.jpg | flash | 182,4 | 205 | 82 | 205 | 2,4 % | 0,7 % | 51,0 | 205 | 205 |
| KTM1d.jpg | luar / matahari | 219,4 | 243 | 117 | 243 | 0,0 % | 82,3 % | 52,6 | 243 | 243 |

**Perbedaan bentuk kurva CDF antara citra paling gelap dan paling terang.**

*Citra paling gelap* (`KTM1a.jpg`, ruangan gelap) memiliki CDF yang **naik sangat cepat di kiri**:
50 % piksel sudah tercapai pada intensitas **35**, dan p90 juga 35 — artinya 90 % piksel bernilai
≤ 35. Hampir seluruh massa histogram menumpuk di rentang 6–35 (18,4 % piksel bahkan ≤ 32), dan
tidak ada satu pun piksel yang ≥ 224. Rentang dinamis yang dipakai hanya sekitar 0–60 dari 0–255,
sehingga kurva CDF praktis mendatar (jenuh di nilai 1,0) sejak intensitas rendah.

*Citra paling terang* (`KTM1d.jpg`, luar ruangan) menunjukkan kebalikannya: CDF **datar panjang di
kiri** lalu menanjak tajam di kanan. Baru pada intensitas **117** (p5) kurva mulai bergerak, 50 %
tercapai pada **243**, dan 82,3 % piksel bernilai ≥ 224. Dengan kata lain sebagian besar piksel
sudah jenuh (mendekati putih) dan detail hanya tersisa di ekor kiri kurva.

**Kaitan dengan kondisi pencahayaan Pertemuan 2.** Data di atas persis mencerminkan cara akuisisi:
`KTM1a` diambil di ruangan gelap sehingga sensor hanya menerima sedikit cahaya (rerata 29,9) dan
seluruh histogram tertekan ke kiri; `KTM1b` dengan lampu ruangan naik ke rerata 119,5 dan mulai
memakai rentang menengah; `KTM1c` memakai flash sehingga muncul lonjakan besar di ujung terang
(0,7 % piksel sudah ≥ 224) tetapi latar masih gelap — terlihat dari p5 yang masih 82 dan std
terbesar (51,0); `KTM1d` di bawah matahari membuat hampir seluruh bidang teroverekspos
(rerata 219,4, p95 = median = 243) sehingga informasi paling banyak hilang karena saturasi.
Kesimpulan praktisnya: keempat citra **belum** memanfaatkan rentang 0–255 secara merata, sehingga
tahap berikutnya (histogram equalization) memang relevan — khususnya untuk `KTM1a` yang rentangnya
paling sempit.

Catatan: pada citra gelap, CDF yang menanjak sangat cepat berarti transformasi ekualisasi akan
"meledakkan" selisih antar intensitas rendah, dan justru itu yang membuat tulisan pada kartu
menjadi terbaca (dibuktikan pada E-2).

In [ ]:
# ---- PERTANYAAN E1 no.3 : KTM1b.jpg dengan PARAM['bins'] bin vs 256 bin ----
nb = PARAM['bins']
h_nb, _ = np.histogram(ktm1b_g, bins=nb, range=(0, 256))
plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1)
plt.imshow(ktm1b_g, cmap='gray'); plt.title(T('KTM1b.jpg - citra asli')); plt.axis('off')
plt.subplot(1, 3, 2)
plt.bar(np.arange(nb), h_nb, width=0.9, color='tab:purple')
plt.title(T('KTM1b.jpg - histogram {} bin (PARAM bins)'.format(nb)))
plt.xlabel('bin intensitas'); plt.ylabel('jumlah piksel')
plt.subplot(1, 3, 3)
plt.bar(np.arange(256), h_ktm1b, width=0.9, color='tab:blue')
plt.title(T('KTM1b.jpg - histogram 256 bin'))
plt.xlabel('intensitas'); plt.ylabel('jumlah piksel')
plt.tight_layout(); plt.show()

def puncak_lokal(h):
    """jumlah puncak lokal pada histogram (naik lalu turun)"""
    h = np.asarray(h, dtype=np.float64)
    return int(np.sum((h[1:-1] > h[:-2]) & (h[1:-1] > h[2:])))

lebar = 256.0 / nb
imax = int(np.argmax(h_nb))
print('lebar satu bin pada {} bin = {:.0f} level intensitas'.format(nb, lebar))
print('puncak histogram {} bin : bin {} (level {}..{}) dengan {} piksel'
      .format(nb, imax, int(imax * lebar), int((imax + 1) * lebar) - 1, int(h_nb.max())))
print('bin terisi : {} dari {} bin ({} bin) | {} dari 256 bin (256 bin)'
      .format(int((h_nb > 0).sum()), nb, nb, int((h_ktm1b > 0).sum())))
print('puncak lokal : {} pada {} bin | {} pada 256 bin'
      .format(puncak_lokal(h_nb), nb, puncak_lokal(h_ktm1b)))
print('level intensitas yang tidak muncul pada 256 bin : {}'.format(int((h_ktm1b == 0).sum())))
print('total piksel {} bin = {} | 256 bin = {}'
      .format(nb, int(h_nb.sum()), int(h_ktm1b.sum())))

## Jawaban PERTANYAAN E1 no.3 — histogram 32 bin vs 256 bin (KTM1b.jpg)

`PARAM['bins'] = 32`, sehingga satu bin berisi **8 level intensitas** (256/32). Kedua histogram
ditampilkan berdampingan pada citra yang sama.

| Ukuran | 32 bin | 256 bin |
|---|---|---|
| Lebar satu bin | 8 level | 1 level |
| Puncak | bin 17 (level 136–143) dengan 505.835 piksel | level 136 |
| Bin/level terisi | 25 dari 32 bin | 195 dari 256 level |
| Jumlah puncak lokal | 5 | 40 |
| Level intensitas yang tidak muncul | — (semua bin gabungan) | 61 level |
| Total piksel | 645.018 | 645.018 |

**Informasi yang hilang ketika jumlah bin dikurangi.** Total piksel tetap sama, jadi tidak ada
data yang hilang secara kuantitatif — yang hilang adalah **resolusi** distribusinya:

1. **Lokasi puncak menjadi kabur.** Pada 256 bin, puncak berada tepat di intensitas 136; pada
   32 bin hanya diketahui bahwa puncaknya ada di suatu tempat antara 136–143. Untuk citra ini
   seluruh selisih 7 level itu tercampur dalam satu batang.
2. **Struktur halus hilang.** Jumlah puncak lokal turun dari 40 menjadi 5. Rincian seperti
   "level 136 jauh lebih banyak daripada 137 dan 138" atau adanya 61 level kosong (celah antar
   nilai) tidak terlihat lagi setelah penggabungan.
3. **Ekor distribusi tertutup.** Pada 256 bin terlihat jelas ada piksel di level rendah dan
   beberapa level terpisah di ujung terang; pada 32 bin keduanya lebur menjadi satu batang besar.

Kapan masing-masing dipakai: histogram 256 bin diperlukan untuk analisis presisi (mis. memeriksa
kekosongan level, membandingkan histogram sebelum/sesudah ekualisasi, atau menghitung LUT),
sedangkan 32 bin atau lebih sedikit berguna untuk melihat **bentuk umum** distribusi
(unimodal/bimodal, condong ke kiri atau kanan) tanpa terganggu derau bin-per-bin. Pada laporan
ini histogram 256 bin tetap dipakai sebagai acuan angka, dan versi `PARAM['bins']` dipakai untuk
menampilkan profil kasar distribusi.

## D.2 Histogram Equalization

Histogram equalization memetakan derajat keabuan lama $r$ ke derajat keabuan baru $s$ melalui
fungsi transformasi

$$s = T(r) = (L-1)\sum_{k=0}^{r} p(k), \qquad p(k) = \frac{n_k}{MN}$$

Implementasi manual pada notebook ini memakai bentuk ekuivalen dari modul,
$K_0 = \mathrm{round}\left(\dfrac{C_i \cdot (2^b - 1)}{w \cdot h}\right)$, dengan $C_i$ = distribusi
kumulatif. Langkahnya: (1) hitung frekuensi setiap intensitas, (2) jumlahkan kumulatif,
(3) normalkan dengan jumlah piksel, (4) terapkan rumus $K_0$ sebagai LUT, (5) petakan kembali ke
citra.

```mermaid
flowchart TD
    A[Start] --> B[Input image]
    B --> C[Hitung frekuensi tiap nilai piksel]
    C --> D[Jumlah kumulatif frekuensi dibagi jumlah piksel]
    D --> E[Normalisasi kumulatif]
    E --> F[Terapkan rumus K0 skala warna]
    F --> G[Transformasi kembali ke bentuk citra]
    G --> H[Histogram hasil equalization]
    G --> I[Citra hasil equalization] --> J[End]
```

### Verifikasi rumus dengan contoh 3-bit modul (Tabel 1, citra 64x64, MN = 4096)

Sebelum dipakai pada citra asli, rumus diuji pada data Tabel 1 modul. Hasilnya (kolom `sk` dihitung
dengan nilai $p(r_k)$ eksak $n_k/4096$):

| $r_k$ | $n_k$ | $p(r_k)$ | CDF | $s_k = 7 \times CDF$ | $s_k$ dibulatkan |
|---|---|---|---|---|---|
| r0 = 0 | 790 | 0,19 | 0,19 | 1,35 | **1** |
| r1 = 1 | 1023 | 0,25 | 0,44 | 3,10 | **3** |
| r2 = 2 | 850 | 0,21 | 0,65 | 4,55 | **5** |
| r3 = 3 | 656 | 0,16 | 0,81 | 5,67 | **6** |
| r4 = 4 | 329 | 0,08 | 0,89 | 6,23 | **6** |
| r5 = 5 | 245 | 0,06 | 0,95 | 6,65 | **7** |
| r6 = 6 | 122 | 0,03 | 0,98 | 6,86 | **7** |
| r7 = 7 | 81 | 0,02 | 1,00 | 7,00 | **7** |

Delapan level lama menyusut menjadi **lima** level baru: **1, 3, 5, 6, 7**; level **0, 2, dan 4
tidak pernah muncul** pada hasil, sedangkan $r_3$ dan $r_4$ sama-sama menjadi 6. Inilah sebabnya
histogram hasil ekualisasi pada citra digital **tidak pernah benar-benar rata**. Angka 1,35 dan
3,10 sedikit berbeda dari 1,33 dan 3,08 pada modul karena modul memakai $p(r_k)$ yang sudah
dibulatkan dua desimal (0,19 dan 0,44), sedangkan program memakai nilai eksak $n_k/4096$; hasil
pembulatannya tetap sama.

In [ ]:
# ---- E-2 langkah 1 : histogram equalization manual (tanpa cv.equalizeHist) ----
def he_lut_manual(gray):
    """LUT ekualisasi manual : K0 = round(Ci * (2^b - 1) / (w * h))"""
    h = np.bincount(gray.ravel(), minlength=256).astype(np.float64)  # frekuensi tiap intensitas
    cdf = np.cumsum(h)                                                # penjumlahan kumulatif Ci
    lut = np.round(cdf * 255.0 / gray.size)                           # (2^b - 1) = 255, w*h = size
    return np.clip(lut, 0, 255).astype(np.uint8)

def he_manual(gray):
    """citra hasil ekualisasi = LUT dipetakan kembali ke setiap piksel"""
    return he_lut_manual(gray)[gray]

# verifikasi rumus dengan contoh 3-bit pada modul (Tabel 1, citra 64x64, MN = 4096)
nk = np.array([790, 1023, 850, 656, 329, 245, 122, 81], dtype=np.float64)
assert nk.sum() == 4096
cdf3 = np.cumsum(nk) / 4096.0
sk = cdf3 * 7
print('%-7s %-7s %-7s %-9s %-12s %s' % ('rk', 'nk', 'p(rk)', 'CDF(rk)', 'sk = 7 x CDF', 'sk dibulatkan'))
for r in range(8):
    print('r%d = %-3d %-7d %-7.2f %-9.2f %-12.2f %d'
          % (r, r, nk[r], nk[r] / 4096, cdf3[r], sk[r], int(np.round(sk[r]))))
print('level lama 0..7 menyusut menjadi level baru :', sorted(set(int(np.round(v)) for v in sk)))
print('level yang tidak pernah muncul pada hasil :', sorted(set(range(8)) - set(int(np.round(v)) for v in sk)))

### E-2 Langkah 1 (Tahap 1) — ekualisasi manual pada lena_lc.jpg

In [ ]:
# ---- Tahap 1 : ekualisasi citra contoh lena_lc.jpg ----
lena_lc = cv.imread(p('lena_lc.jpg'), cv.IMREAD_GRAYSCALE)
he_lc_manual = he_manual(lena_lc)
he_lc_cv = cv.equalizeHist(lena_lc)
d_lc = np.abs(he_lc_manual.astype(np.int16) - he_lc_cv.astype(np.int16))

print('lena_lc.jpg :', lena_lc.shape)
print('rerata sebelum %.1f | setelah manual %.1f | setelah cv.equalizeHist %.1f'
      % (lena_lc.mean(), he_lc_manual.mean(), he_lc_cv.mean()))
print('std sebelum %.1f | setelah manual %.1f | setelah cv %.1f'
      % (lena_lc.std(), he_lc_manual.std(), he_lc_cv.std()))
print('max |manual - cv.equalizeHist| = %d | jumlah piksel berbeda = %d (%.3f%%)'
      % (d_lc.max(), int((d_lc > 0).sum()), (d_lc > 0).mean() * 100))

plt.figure(figsize=(15, 8))
for i, (c, t) in enumerate([(lena_lc, 'asli'), (he_lc_manual, 'HE manual'), (he_lc_cv, 'HE cv.equalizeHist')], 1):
    plt.subplot(2, 3, i); plt.imshow(c, cmap='gray'); plt.title(T('lena_lc.jpg - ' + t), fontsize=9); plt.axis('off')
    plt.subplot(2, 3, i + 3)
    plt.hist(c.ravel(), bins=256, range=(0, 256), color='tab:blue', lw=0)
    plt.title(T('histogram ' + t), fontsize=9); plt.xlabel('intensitas'); plt.ylabel('jumlah piksel')
plt.tight_layout(); plt.show()

**Hasil tahap 1 (`lena_lc.jpg`):** citra contoh berkontras rendah (std hanya 12,0) berubah
menjadi jauh lebih menyebar setelah ekualisasi manual — std naik ke 73,6 dan rerata bergeser
136,7 → 130,6. Bandingkan dengan `lena.jpg` yang std-nya sudah 47,9 sejak awal: itulah alasan modul memilih
`lena_lc.jpg` untuk percobaan ini, karena efek ekualisasi baru terlihat jelas pada citra
berkontras rendah.

### E-2 Langkah 1 (Tahap 2) — ekualisasi manual pada KTM1a.jpg (ruangan gelap)

In [ ]:
# ---- Tahap 2 : ekualisasi citra KTM1a.jpg (ruangan gelap) ----
ktm1a_g = cv.imread(p('KTM1a.jpg'), cv.IMREAD_GRAYSCALE)
he_ktm_manual = he_manual(ktm1a_g)
he_ktm_cv = cv.equalizeHist(ktm1a_g)
d_ktm = np.abs(he_ktm_manual.astype(np.int16) - he_ktm_cv.astype(np.int16))

print('KTM1a.jpg :', ktm1a_g.shape)
print('rerata sebelum %.1f | setelah manual %.1f | setelah cv.equalizeHist %.1f'
      % (ktm1a_g.mean(), he_ktm_manual.mean(), he_ktm_cv.mean()))
print('std sebelum %.1f | setelah manual %.1f | setelah cv %.1f'
      % (ktm1a_g.std(), he_ktm_manual.std(), he_ktm_cv.std()))
print('max |manual - cv.equalizeHist| = %d | jumlah piksel berbeda = %d (%.3f%%)'
      % (d_ktm.max(), int((d_ktm > 0).sum()), (d_ktm > 0).mean() * 100))

plt.figure(figsize=(15, 8))
for i, (c, t) in enumerate([(ktm1a_g, 'asli'), (he_ktm_manual, 'HE manual'), (he_ktm_cv, 'HE cv.equalizeHist')], 1):
    plt.subplot(2, 3, i); plt.imshow(c, cmap='gray'); plt.title(T('KTM1a.jpg - ' + t), fontsize=9); plt.axis('off')
    plt.subplot(2, 3, i + 3)
    plt.hist(c.ravel(), bins=256, range=(0, 256), color='tab:green', lw=0)
    plt.title(T('histogram ' + t), fontsize=9); plt.xlabel('intensitas'); plt.ylabel('jumlah piksel')
plt.tight_layout(); plt.show()

Hasil tahap 2 jauh lebih dramatis: rerata **29,9 → 201,6** dan std **11,4 → 85,5**. Rentang
intensitas yang semula hanya terpakai 0–60 kini memakai hampir seluruh skala 0–255. Pada
`KTM1a.jpg` tulisan kartu sebenarnya sudah terbaca sejak awal, tetapi dengan kontras rendah
(selisih rerata kelompok teks dan latar hanya 32,5 menurut ambang Otsu); setelah ekualisasi
selisih itu menjadi 214,0, sehingga huruf "SINGGIH WAHYU PERMANA" dan "NIM : 244107020102"
tampak hitam tegas di atas latar putih — lihat angka Otsu pada jawaban PERTANYAAN E2 no.1.

### E-2 Langkah 2 — manual vs `cv.equalizeHist` dan penyebab selisihnya

In [ ]:
# ---- selisih manual vs cv.equalizeHist : cari penyebabnya ----
def he_lut_opencv(gray):
    """LUT yang meniru cv.equalizeHist: LUT[imin] = 0, skala 255/(n - hist[imin])"""
    h = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    n = float(gray.size)
    imin = int(np.argmax(h > 0))          # intensitas pertama yang muncul
    cdf = np.cumsum(h)
    scale = 255.0 / (n - h[imin])         # pembagi OpenCV
    lut = np.zeros(256, dtype=np.float64)
    lut[imin + 1:] = (cdf[imin + 1:] - cdf[imin]) * scale
    return np.clip(np.round(lut), 0, 255).astype(np.uint8)

for nama, g in [('lena_lc.jpg', lena_lc), ('KTM1a.jpg', ktm1a_g)]:
    a = he_lut_manual(g)[g]
    b = he_lut_opencv(g)[g]
    c = cv.equalizeHist(g)
    h = np.bincount(g.ravel(), minlength=256)
    imin = int(np.argmax(h > 0))
    print('== %s ==' % nama)
    print('   intensitas terkecil yang muncul (imin) = %d dengan %d piksel' % (imin, h[imin]))
    print('   LUT manual (Gonzalez)  : LUT[%d] = %d | LUT[255] = %d'
          % (imin, he_lut_manual(g)[imin], he_lut_manual(g)[255]))
    print('   LUT tiruan OpenCV      : LUT[%d] = %d | LUT[255] = %d'
          % (imin, he_lut_opencv(g)[imin], he_lut_opencv(g)[255]))
    print('   max |manual - cv2|           = %d' % np.abs(a.astype(np.int16) - c.astype(np.int16)).max())
    print('   max |tiruan OpenCV - cv2|    = %d' % np.abs(b.astype(np.int16) - c.astype(np.int16)).max())

## Jawaban E-2 Langkah 2 — selisih manual vs `cv.equalizeHist`

| Berkas | max \|manual − `cv.equalizeHist`\| | Piksel berbeda | Rerata sebelum → manual → cv2 | Std sebelum → manual → cv2 |
|---|---|---|---|---|
| `lena_lc.jpg` (tahap 1) | **0** | 0 (0,000 %) | 136,7 → 130,6 → 130,6 | 12,0 → 73,6 → 73,6 |
| `KTM1a.jpg` (tahap 2) | **5** | 134.520 (20,855 %) | 29,9 → 201,6 → 200,7 | 11,4 → 85,5 → 87,2 |

Pada citra contoh selisihnya **nol** — implementasi manual sudah benar. Pada `KTM1a.jpg`
selisih maksimum 5 level dan berbeda pada 20,9 % piksel, jadi penyebabnya bukan kesalahan rumus
melainkan perbedaan **definisi LUT**, dan penyebab itu dibuktikan langsung oleh program:

| Berkas | intensitas terkecil yang muncul (`imin`) | LUT manual (Gonzalez) | LUT tiruan OpenCV | max \|manual − cv2\| | max \|tiruan − cv2\| |
|---|---|---|---|---|---|
| `lena_lc.jpg` | 108 (hanya **1 piksel**) | LUT[108] = 0, LUT[255] = 255 | LUT[108] = 0 | 0 | 0 |
| `KTM1a.jpg` | **0** (12.633 piksel) | **LUT[0] = 5**, LUT[255] = 255 | **LUT[0] = 0** | **5** | **0** |

**Penyebabnya:**

1. Rumus Gonzalez pada modul memetakan tiap intensitas $r$ dengan
   $s = \mathrm{round}\big((L-1)\,CDF(r) / (MN)\big)$, sehingga untuk $r = 0$ pada `KTM1a`
   menghasilkan $s = \mathrm{round}(255 \times 12633/645018) = \mathbf{5}$. Intensitas 0 tetap
   mendapat nilai 5, bukan 0.
2. `cv.equalizeHist` memakai varian yang **menggeser skala** agar intensitas terkecil yang
   benar-benar muncul dipetakan tepat ke 0:
   $LUT[r] = \mathrm{round}\big((CDF(r) - CDF(imin)) \cdot \dfrac{255}{MN - n_{imin}}\big)$.
   Untuk `KTM1a`, $imin = 0$ sehingga $LUT[0] = 0$ dan seluruh LUT tergeser turun 5 level.
3. Buktinya terlihat pada tabel: begitu LUT tiruan OpenCV (yang menggeser $imin$) dipakai,
   selisih terhadap `cv.equalizeHist` menjadi **0 level** untuk kedua berkas, sedangkan LUT
   Gonzalez berbeda 5 level. Karena itu perbedaan 20,9 % piksel pada `KTM1a` **bukan bug**,
   melainkan konsekuensi definisi skala — dan justru muncul pada citra yang punya banyak piksel
   bernilai 0 (ruangan gelap), sementara pada `lena_lc.jpg` (hanya 1 piksel di $imin$) kedua
   definisi berimpit sehingga selisihnya nol.

### E-2 Langkah 3 — ekualisasi pada citra berwarna

Modul menyebut dua cara: **cara 1** mengekualisasi kanal biru, hijau, dan merah satu per satu,
sedangkan **cara 2** memisahkan kecerahan dari informasi warna (hanya kanal terang yang
diekualisasi). Varian setara kanal terang dicoba sekaligus pada YCrCb (kanal **Y**), HSV (kanal
**V**), dan LAB (kanal **L**). Perbedaan kedua cara tidak cukup dinilai dengan mata, sehingga
program mengukur **pergeseran Hue** rata-rata:

$$\bar{\Delta H} = \frac{1}{MN}\sum \min\big(|h_1 - h_2|,\ 180 - |h_1 - h_2|\big) \times 2$$

Karena Hue OpenCV bersifat melingkar pada rentang 0–179, selisih dihitung lewat jalur terpendek
lalu dikali 2 agar bersatuan derajat (0–358°). Sebelum dipakai, alat ukur ini dikalibrasi dengan
mengukur pergeseran Hue akibat konversi bolak-balik BGR → ruang warna → BGR tanpa ekualisasi
apa pun — itulah batas bawah derau pengukurannya.

In [ ]:
# ---- alat ukur pergeseran warna : kanal Hue (melingkar 0..179) ----
def geser_hue(asli, hasil):
    """rerata pergeseran Hue dalam derajat (Hue OpenCV 0..179 -> x2 menjadi 0..358)"""
    h1 = cv.cvtColor(asli, cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    h2 = cv.cvtColor(hasil, cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    d = np.abs(h1 - h2)
    d = np.minimum(d, 180 - d)          # Hue melingkar 0..179 -> ambil jalur terpendek
    return d.mean()

# pembuktian : konversi bolak-balik BGR -> ruang warna -> BGR pun sudah menggeser Hue
print('Pergeseran Hue murni akibat konversi bolak-balik (tanpa ekualisasi apa pun):')
print('%-12s %14s %14s %14s' % ('berkas', 'YCrCb (der)', 'HSV (der)', 'LAB (der)'))
for _f in ['lena.jpg', 'KTM1d.jpg', 'KTM1a.jpg']:
    _b = cv.imread(p(_f))
    _y = cv.cvtColor(cv.cvtColor(_b, cv.COLOR_BGR2YCrCb), cv.COLOR_YCrCb2BGR)
    _h = cv.cvtColor(cv.cvtColor(_b, cv.COLOR_BGR2HSV), cv.COLOR_HSV2BGR)
    _l = cv.cvtColor(cv.cvtColor(_b, cv.COLOR_BGR2LAB), cv.COLOR_LAB2BGR)
    print('%-12s %14.3f %14.3f %14.3f'
          % (_f, geser_hue(_b, _y) * 2, geser_hue(_b, _h) * 2, geser_hue(_b, _l) * 2))
    print('%-12s selisih nilai piksel maksimum: YCrCb %d | HSV %d | LAB %d'
          % ('', np.abs(_b.astype(np.int16) - _y.astype(np.int16)).max(),
             np.abs(_b.astype(np.int16) - _h.astype(np.int16)).max(),
             np.abs(_b.astype(np.int16) - _l.astype(np.int16)).max()))
print()

def ekualisasi_warna(berkas, keterangan):
    """cara 1 (B,G,R terpisah) vs cara 2 (kanal terang saja) pada satu berkas citra"""
    bgr = cv.imread(p(berkas))
    kanal = list(cv.split(bgr))
    for i in range(3):
        kanal[i] = cv.equalizeHist(kanal[i])
    he_rgb = cv.merge(kanal)

    ycrcb = cv.cvtColor(bgr, cv.COLOR_BGR2YCrCb)
    y, cr, cb = cv.split(ycrcb)
    he_y = cv.cvtColor(cv.merge([cv.equalizeHist(y), cr, cb]), cv.COLOR_YCrCb2BGR)

    hsv = cv.cvtColor(bgr, cv.COLOR_BGR2HSV)
    h, sat, v = cv.split(hsv)
    he_v = cv.cvtColor(cv.merge([h, sat, cv.equalizeHist(v)]), cv.COLOR_HSV2BGR)

    lab = cv.cvtColor(bgr, cv.COLOR_BGR2LAB)
    l, a, b = cv.split(lab)
    he_l = cv.cvtColor(cv.merge([cv.equalizeHist(l), a, b]), cv.COLOR_LAB2BGR)

    judul = ['Asli', 'Ekualisasi B, G, R', 'Kanal Y', 'Kanal V', 'Kanal L']
    citra = [bgr, he_rgb, he_y, he_v, he_l]

    plt.figure(figsize=(18, 4.5))
    for i, (c, t) in enumerate(zip(citra, judul), 1):
        plt.subplot(1, 5, i)
        plt.imshow(cv.cvtColor(c, cv.COLOR_BGR2RGB))
        plt.title(T(t), fontsize=9)
        plt.axis('off')
    plt.suptitle(T('%s - %s' % (berkas, keterangan)), fontsize=11)
    plt.tight_layout(); plt.show()

    print('%-22s %18s %18s %s' % ('Cara', 'geser Hue (derajat)', 'rerata terang', 'catatan'))
    hasil = []
    for t, c in zip(judul, citra):
        gh = geser_hue(bgr, c) * 2
        terang = cv.cvtColor(c, cv.COLOR_BGR2GRAY).mean()
        hasil.append((t, gh, terang))
        print('%-22s %18.2f %18.1f' % (t, gh, terang))
    return hasil

print('########## TAHAP 1 : citra contoh ##########')
hasil_lena = ekualisasi_warna('lena.jpg', 'tahap 1 - citra contoh')

In [ ]:
print('########## TAHAP 2 : citra KTM sendiri ##########')
hasil_ktmd = ekualisasi_warna('KTM1d.jpg', 'tahap 2 - KTM cahaya matahari')

In [ ]:
# ---- pertanyaan E-2 no.4 : CLAHE pada kanal terang (KTM1d.jpg) ----
def clahe_kanal(berkas, ruang, clip, tile):
    """ekualisasi kanal terang dengan CLAHE, kanal warna tidak disentuh"""
    bgr = cv.imread(p(berkas))
    cl = cv.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
    if ruang == 'Y':
        konv, balik = cv.COLOR_BGR2YCrCb, cv.COLOR_YCrCb2BGR
    elif ruang == 'V':
        konv, balik = cv.COLOR_BGR2HSV, cv.COLOR_HSV2BGR
    else:
        konv, balik = cv.COLOR_BGR2LAB, cv.COLOR_LAB2BGR
    spasi = cv.cvtColor(bgr, konv)
    k1, k2, k3 = cv.split(spasi)
    return cv.cvtColor(cv.merge([cl.apply(k1), k2, k3]), balik)

bgr_d = cv.imread(p('KTM1d.jpg'))
hasil_clahe = {}
for ruang in ['Y', 'V', 'L']:
    hasil_clahe[ruang] = clahe_kanal('KTM1d.jpg', ruang, PARAM['clip'], PARAM['tile'])

# pembanding : ekualisasi biasa pada kanal yang sama
y, cr, cb = cv.split(cv.cvtColor(bgr_d, cv.COLOR_BGR2YCrCb))
he_y_biasa = cv.cvtColor(cv.merge([cv.equalizeHist(y), cr, cb]), cv.COLOR_YCrCb2BGR)

# cara 1 (B, G, R terpisah) pada berkas yang sama, untuk pembanding kesetiaan warna
kanal_d = list(cv.split(bgr_d))
for _i in range(3):
    kanal_d[_i] = cv.equalizeHist(kanal_d[_i])
he_rgb_d = cv.merge(kanal_d)

# dasar penilaian kesetiaan warna : bar oranye judul kartu (baris 0-25, kolom 0-1010)
# area ini murni oranye tanpa teks, sehingga pergeseran warna langsung terukur
baris_bar, kolom_bar = slice(0, 26), slice(0, 1011)
warna_asli = bgr_d[baris_bar, kolom_bar].astype(np.float64).reshape(-1, 3).mean(axis=0)
hue_asli = cv.cvtColor(bgr_d, cv.COLOR_BGR2HSV)[baris_bar, kolom_bar][:, :, 0].astype(np.int16)

def setia_bar(c):
    """selisih warna bar oranye terhadap citra asli (B, G, R) + pergeseran Hue-nya"""
    v = c[baris_bar, kolom_bar].astype(np.float64).reshape(-1, 3).mean(axis=0)
    hb = cv.cvtColor(c, cv.COLOR_BGR2HSV)[baris_bar, kolom_bar][:, :, 0].astype(np.int16)
    d = np.abs(hue_asli - hb); d = np.minimum(d, 180 - d)
    return v, float(np.abs(v - warna_asli).mean()), float(d.mean() * 2)

judul = ['Asli', 'HE kanal Y (biasa)', 'CLAHE Y clip %.1f' % PARAM['clip'],
         'CLAHE V clip %.1f' % PARAM['clip'], 'CLAHE L clip %.1f' % PARAM['clip']]
citra = [bgr_d, he_y_biasa, hasil_clahe['Y'], hasil_clahe['V'], hasil_clahe['L']]
plt.figure(figsize=(18, 4.5))
for i, (c, t) in enumerate(zip(citra, judul), 1):
    plt.subplot(1, 5, i)
    plt.imshow(cv.cvtColor(c, cv.COLOR_BGR2RGB)); plt.title(T(t), fontsize=9); plt.axis('off')
plt.suptitle(T('KTM1d.jpg - CLAHE clip %.1f, tile %dx%d' % (PARAM['clip'], PARAM['tile'], PARAM['tile'])), fontsize=11)
plt.tight_layout(); plt.show()

print()
print('warna bar biru tua asli (B, G, R) :', warna_asli.round(1))
print('%-22s %-26s %12s %12s' % ('cara', 'warna bar sesudah (B, G, R)', 'delta rerata', 'geser Hue bar'))
for t, c in zip(judul + ['Ekualisasi B, G, R'], citra + [he_rgb_d]):
    v, dv, dh = setia_bar(c)
    print('%-22s %-26s %12.1f %12.2f' % (t, str(v.round(1)), dv, dh))
print()

print('%-24s %16s %16s' % ('Cara', 'geser Hue (derajat)', 'rerata terang'))
for t, c in zip(judul, citra):
    print('%-24s %16.2f %16.1f' % (t, geser_hue(bgr_d, c) * 2, cv.cvtColor(c, cv.COLOR_BGR2GRAY).mean()))

### E-2 Langkah 3 — ekualisasi citra berwarna: cara 1 (B, G, R terpisah) vs cara 2 (kanal terang)

Sebelum membandingkan, program mengukur **pergeseran Hue murni akibat konversi bolak-balik**
(tanpa ekualisasi sama sekali) sebagai batas bawah derau pengukuran:

| Berkas | YCrCb (derajat) | HSV (derajat) | LAB (derajat) | Selisih nilai piksel maksimum (YCrCb/HSV/LAB) |
|---|---|---|---|---|
| `lena.jpg` | 0,041 | 0,013 | 0,367 | 1 / 3 / 4 |
| `KTM1d.jpg` | 0,071 | 1,553 | 1,658 | 1 / 4 / 6 |
| `KTM1a.jpg` | 0,354 | 0,638 | 2,430 | 1 / 1 / 5 |

Artinya konversi BGR → ruang warna → BGR saja sudah dapat menggeser Hue hingga beberapa derajat
(maksimum 6 level piksel), sehingga angka pergeseran Hue hasil ekualisasi harus dibaca relatif
terhadap nilai-nilai kecil ini.

**Tahap 1 (citra contoh `lena.jpg`):**

| Cara | Geser Hue (derajat) | Rerata terang |
|---|---|---|
| Citra asli | 0,00 | 124,2 |
| Ekualisasi kanal B, G, R | **75,01** | 128,3 |
| Kanal Y (YCrCb) | 1,71 | 128,0 |
| Kanal V (HSV) | 0,65 | **91,6** |
| Kanal L (LAB) | 2,65 | 121,8 |

**Tahap 2 (citra KTM sendiri `KTM1d.jpg`):**

| Cara ekualisasi | Rata-rata pergeseran Hue (derajat) | Rerata kecerahan sesudah | Catatan pengamatan visual |
|---|---|---|---|
| Citra asli | — | 219,4 | acuan; kartu terang, banner biru tua, latar biru muda |
| Ekualisasi kanal B, G, R | **20,33** | 198,7 | warna kartu berubah: banner biru tua menjadi abu gelap, latar biru muda menjadi pucat, kertas putih tampak keabu-abuan |
| Kanal Y (YCrCb) | 0,76 | 199,0 | warna paling setia; yang berubah hanya kontras terang |
| Kanal V (HSV) | 2,68 | 196,4 | hampir setia, bagian biru tua sedikit lebih gelap dari aslinya |
| Kanal L (LAB) | 0,86 | 197,7 | setia; hasil terang lebih lembut daripada kanal Y |

Pengukuran pendukung pada **bar banner biru tua** kartu (baris 0–25, murni berwarna tanpa teks;
warna asli B, G, R = 216,0 / 123,1 / 67,0):

| Cara | Warna bar sesudah (B, G, R) | Selisih rerata | Geser Hue bar |
|---|---|---|---|
| Asli | 216,0 / 123,1 / 67,0 | 0,0 | 0,00° |
| Ekualisasi B, G, R | 38,3 / 34,0 / 33,4 | **100,1** | **32,14°** |
| HE kanal Y (biasa) | 138,8 / 45,8 / 0,0 | 73,8 | 2,27° |
| CLAHE kanal Y | 211,6 / 118,6 / 61,6 | **4,8** | **0,00°** |
| CLAHE kanal V | 216,0 / 67,2 / 84,7 | 24,5 | 29,16° |
| CLAHE kanal L | 209,8 / 117,8 / 59,3 | 6,4 | 1,20° |

**1. Cara mana yang menghasilkan pergeseran Hue paling besar?**
Ekualisasi **kanal B, G, R terpisah** (cara 1): **20,33 derajat** pada `KTM1d.jpg`, dan jauh lebih
ekstrem pada citra contoh `lena.jpg` yaitu **75,01 derajat**. Angka ini puluhan kali lebih besar
daripada cara kanal terang dan daripada derau konversi bolak-balik (0,07–1,66 derajat). Pada bar
banner kartu, cara 1 mengubah warna 216/123/67 menjadi 38,3/34,0/33,4 (selisih rerata **100,1**,
geser Hue **32,14°**) — banner biru tua berubah menjadi abu gelap. Sebabnya jelas: setiap kanal
dinormalkan dengan CDF-nya sendiri, sehingga rasio antar kanal (yang menentukan Hue) rusak total.
Sebaliknya, ekualisasi pada kanal terang hanya menyentuh satu kanal sehingga Hue relatif terjaga.

**2. Mengapa ekualisasi kanal terang tetap menghasilkan pergeseran Hue yang tidak persis nol?**
Karena perjalanan BGR → ruang warna → BGR **tidak bolak-balik sempurna** pada aritmetika 8-bit.
Saat citra diubah ke YCrCb/HSV/LAB, kanal krominans (Cr/Cb, H/S, a/b) disimpan sebagai bilangan
bulat 8-bit hasil pembulatan; ketika citra dikembalikan ke BGR, konversi balik melakukan
pembulatan dan **pemotongan** (clipping) lagi, sehingga nilai kanal B, G, R hasilnya bergeser
1–6 level dari aslinya. Pergeseran kecil pada kanal B, G, R inilah yang muncul sebagai geser Hue
beberapa derajat. Batas bawahnya sudah terukur pada tabel konversi bolak-balik: `KTM1d.jpg`
memberi 0,071° (YCrCb), 1,553° (HSV), dan 1,658° (LAB) **tanpa ekualisasi apa pun**. Karena itu
angka 0,76° (Y), 2,68° (V), dan 0,86° (L) pada tahap 2 sebagian besar berasal dari pembulatan
konversi, bukan dari ekualisasi. Bukti tambahan: pada CLAHE kanal Y, pergeseran Hue citra penuh
turun ke 0,13° dan pergeseran Hue bar banner tepat **0,00°** — lebih kecil daripada derau
konversi biasa, karena CLAHE mengubah kanal Y lebih halus sehingga hasil konversi baliknya
kebetulan lebih dekat ke aslinya.

**3. Di antara kanal Y, V, dan L, mana yang paling sesuai untuk foto KTM ini?**
**Kanal Y (YCrCb)** paling sesuai. Alasannya bertumpu pada angka, bukan hanya penglihatan:

* Pergeseran Hue citra penuh paling kecil (0,76° pada tahap 2 dan 1,71° pada tahap 1), dengan
  warna bar banner hanya bergeser **2,27°** dan pada versi CLAHE-nya tepat **0,00°** dengan
  selisih warna bar 4,8 — nyaris tak terlihat.
* Kecerahan akhirnya paling dekat dengan aslinya (199,0 dari 219,4) sehingga kartu tidak menjadi
  terlalu gelap.
* Kanal V justru paling berisiko untuk citra ini: pada `KTM1d.jpg` pergeseran Hue-nya terbesar di
  antara ketiga kanal terang (2,68° penuh; **8,73°** saat memakai CLAHE) dan pada bar banner
  CLAHE V pergeserannya mencapai **29,16°** dengan warna berubah dari 216/123/67 menjadi
  216/67/85 — biru tua menjadi keunguan. Sebabnya, pada HSV informasi kecerahan (V) masih
  bercampur dengan saturasi, sehingga mengubah V ikut mengubah S dan akhirnya H.
* Kanal L berada di tengah: pergeseran Hue kecil (0,86°, CLAHE 1,72°) dan warna bar cukup setia
  (selisih 6,4), tetapi hasilnya sedikit lebih lembut/kurang kontras daripada kanal Y.
* **Bagian citra yang menjadi dasar penilaian:** bar banner biru tua kartu (baris 0–25) dan
  tulisan "SINGGIH WAHYU PERMANA" beserta "NIM : 244107020102" (sekitar baris 150–300, kolom
  20–560). Pada bar banner, pergeseran warna terbukti kuantitatif (tabel di atas), sedangkan pada
  area tulisan terlihat bahwa kanal Y dan L mempertahankan warna hitam-tegas tulisan, sementara
  cara B, G, R membuat banner dan latar berubah warna.

**4. CLAHE pada kanal terang (`clipLimit = PARAM['clip'] = 2.0`, `tileGridSize = (4, 4)`):**

| Cara | Geser Hue (derajat) | Rerata terang |
|---|---|---|
| Citra asli | 0,00 | 219,4 |
| HE kanal Y (biasa) | 0,76 | 199,0 |
| **CLAHE Y clip 2,0** | **0,13** | 216,0 |
| CLAHE V clip 2,0 | 8,73 | 214,8 |
| CLAHE L clip 2,0 | 1,72 | 216,3 |

Dibanding ekualisasi biasa pada kanal yang sama, CLAHE **menurunkan pergeseran Hue**
(0,76° → 0,13° untuk kanal Y) dan **mempertahankan kecerahan** hampir persis seperti aslinya
(199,0 → 216,0 dari 219,4). Ini masuk akal karena CLAHE membatasi penguatan kontras per blok
lokal sehingga tidak memaksa seluruh rentang dinamis terpakai; karena perubahannya lebih halus,
galat pembulatan saat konversi balik ke BGR pun lebih kecil.

In [ ]:
# ---- PERTANYAAN E2 no.1 : ekualisasi global pada KTM1a.jpg + PSNR ----
def psnr(a, b):
    """PSNR antara dua citra 8-bit"""
    a = a.astype(np.float64); b = b.astype(np.float64)
    mse = np.mean((a - b) ** 2)
    return float('inf') if mse == 0 else 10 * math.log10(255.0 ** 2 / mse)

def entropi(g):
    """entropi histogram citra (bit) sebagai ukuran kandungan informasi"""
    h = np.bincount(g.ravel(), minlength=256).astype(np.float64)
    pj = h / h.sum()
    pj = pj[pj > 0]
    return float(-(pj * np.log2(pj)).sum())

p_man_cv = psnr(he_ktm_manual, he_ktm_cv)
p_orig_man = psnr(ktm1a_g, he_ktm_manual)
p_orig_cv = psnr(ktm1a_g, he_ktm_cv)

print('KTM1a.jpg (grayscale)')
print('PSNR asli vs HE manual        : %.2f dB' % p_orig_man)
print('PSNR asli vs cv.equalizeHist  : %.2f dB' % p_orig_cv)
print('PSNR HE manual vs HE cv2      : %.2f dB' % p_man_cv)
print('rerata %.1f -> %.1f | std %.1f -> %.1f | entropi %.3f bit -> %.3f bit'
      % (ktm1a_g.mean(), he_ktm_manual.mean(), ktm1a_g.std(), he_ktm_manual.std(),
         entropi(ktm1a_g), entropi(he_ktm_manual)))

# ukuran tambahan untuk menjawab mengapa PSNR turun padahal citra lebih jelas
h_asli = np.bincount(ktm1a_g.ravel(), minlength=256)
h_he = np.bincount(he_ktm_manual.ravel(), minlength=256)
mse = float(np.mean((ktm1a_g.astype(np.float64) - he_ktm_manual.astype(np.float64)) ** 2))
print('level intensitas terisi : asli %d -> HE %d' % (int((h_asli > 0).sum()), int((h_he > 0).sum())))
print('porsi piksel pada level terbanyak : asli %.1f%% (level %d) -> HE %.1f%% (level %d)'
      % (h_asli.max() / ktm1a_g.size * 100, int(h_asli.argmax()),
         h_he.max() / ktm1a_g.size * 100, int(h_he.argmax())))
print('MSE asli vs HE = %.1f (RMSE %.1f level) -> PSNR = 10 log10(255^2 / MSE) = %.2f dB'
      % (mse, math.sqrt(mse), psnr(ktm1a_g, he_ktm_manual)))

def pisah_otsu(g, roi=(slice(150, 300), slice(20, 560))):
    """selisih rerata kelompok terang dan gelap pada area nama/NIM (ambang Otsu)"""
    pk = g[roi]
    t, _ = cv.threshold(pk, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    terang = pk[pk >= t].astype(np.float64)
    gelap = pk[pk < t].astype(np.float64)
    return int(t), float(terang.mean() - gelap.mean())

t0_, s0_ = pisah_otsu(ktm1a_g)
t1_, s1_ = pisah_otsu(he_ktm_manual)
print('pemisahan teks-latar (Otsu) pada area nama/NIM : asli %.1f (ambang %d) -> HE %.1f (ambang %d)'
      % (s0_, t0_, s1_, t1_))

plt.figure(figsize=(16, 8))
panel = [(ktm1a_g, 'asli'), (he_ktm_manual, 'HE manual'), (he_ktm_cv, 'HE cv.equalizeHist')]
for i, (c, t) in enumerate(panel, 1):
    plt.subplot(2, 3, i); plt.imshow(c, cmap='gray')
    plt.title(T('KTM1a.jpg - ' + t), fontsize=9); plt.axis('off')
    plt.subplot(2, 3, i + 3)
    plt.hist(c.ravel(), bins=256, range=(0, 256), color='tab:orange', lw=0)
    plt.title(T('histogram ' + t), fontsize=9); plt.xlabel('intensitas'); plt.ylabel('jumlah piksel')
plt.tight_layout(); plt.show()

# bagian teks nama/NIM pada kartu, dipakai menilai keterbacaan
r0, r1, c0, c1 = 150, 300, 20, 560
plt.figure(figsize=(16, 5))
for i, (c, t) in enumerate(panel, 1):
    plt.subplot(1, 3, i)
    plt.imshow(c[r0:r1, c0:c1], cmap='gray')
    plt.title(T('%s - potongan teks' % t), fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

## Jawaban PERTANYAAN E2 no.1 — ekualisasi global pada KTM1a.jpg dan PSNR

**(a) dan (b)** Ekualisasi manual dan `cv.equalizeHist` diterapkan pada `KTM1a.jpg` grayscale,
lalu citra asli, kedua hasil ekualisasi, dan histogramnya ditampilkan dalam satu layout (termasuk
potongan area tulisan nama/NIM agar keterbacaan terlihat langsung).

**(c) PSNR antara citra asli dan citra hasil ekualisasi:**

| Pasangan yang diukur | PSNR |
|---|---|
| asli vs ekualisasi **manual** | **2,68 dB** |
| asli vs `cv.equalizeHist` | **2,68 dB** |
| ekualisasi manual vs `cv.equalizeHist` | 42,36 dB |

Angka pendukung: rerata 29,9 → 201,6; std 11,4 → 85,5; MSE = 35.111,8 sehingga
RMSE = 187,4 level; PSNR = 10·log₁₀(255²/35111,8) = 2,68 dB. PSNR antara kedua hasil ekualisasi
sangat tinggi (42,36 dB) — artinya keduanya praktis citra yang sama, sehingga angka 2,68 dB
memang murni menggambarkan perbedaan asli vs hasil, bukan derau implementasi.

**Mengapa PSNR justru turun padahal citra terlihat lebih jelas?**
Karena PSNR mengukur **kedekatan nilai piksel** terhadap citra asli, bukan keterbacaan. Pada
`KTM1a.jpg`, 75,3 % piksel asli menumpuk pada satu nilai saja (level 35, ruangan gelap), dan
setelah ekualisasi tumpukan itu berpindah ke level 245. Perpindahan besar pada mayoritas piksel
inilah yang menghasilkan MSE besar (RMSE 187,4 level) sehingga PSNR kecil. Dengan kata lain,
semakin besar perbaikan kontras yang diberikan (selisih rerata asli vs hasil 171,7 level), semakin
kecil PSNR-nya — PSNR menghukum perubahan, padahal di sini perubahan itulah tujuannya. Ukuran lain
menunjukkan perbaikan nyata:

| Ukuran | Asli | Setelah HE | Arah |
|---|---|---|---|
| Pemisahan teks–latar (Otsu) pada area nama/NIM | 32,5 (ambang 18) | **214,0** (ambang 53) | naik 6,6 kali → teks jauh lebih kontras |
| Std (rentang dinamis yang dipakai) | 11,4 | 85,5 | naik 7,5 kali |
| Entropi histogram | 1,619 bit | 1,543 bit | **sedikit turun** |
| Jumlah level intensitas terisi | 59 | 20 | turun (level menyatu) |

**Kesimpulan: PSNR tidak layak dipakai untuk menilai keterbacaan sebuah citra.** PSNR cocok
untuk mengukur kualitas *restorasi* (mis. hasil denoising/kompresi harus sedekat mungkin dengan
aslinya), tetapi pada *peningkatan kontras* justru berbanding terbalik dengan manfaat visual.
Indikator yang lebih tepat untuk keterbacaan adalah kontras lokal / pemisahan teks–latar seperti
angka Otsu di atas (32,5 → 214,0), rentang dinamis (std), atau penilaian visual pada area tulisan.

In [ ]:
# ---- PERTANYAAN E2 no.2 : CLAHE lokal pada KTM1a.jpg ----
def local_std(g, k=31):
    """peta simpangan baku lokal (ukuran kekasaran/noise) dengan box filter"""
    gf = g.astype(np.float32)
    m = cv.boxFilter(gf, -1, (k, k))
    m2 = cv.boxFilter(gf * gf, -1, (k, k))
    return np.sqrt(np.clip(m2 - m * m, 0, None))

sd_asli = local_std(ktm1a_g)
pad = 40
inner = sd_asli[pad:-pad, pad:-pad]
iy, ix = np.unravel_index(int(np.argmin(inner)), inner.shape)
iy += pad; ix += pad
print('titik paling datar pada KTM1a.jpg : baris %d, kolom %d (local std asli = %.2f)'
      % (iy, ix, sd_asli[iy, ix]))

clips = sorted(set([1.0, 1.5, PARAM['clip'], 3.0]))
hasil_clahe_ktm = {}
for cl in clips:
    c = cv.createCLAHE(clipLimit=cl, tileGridSize=(PARAM['tile'], PARAM['tile']))
    hasil_clahe_ktm[cl] = c.apply(ktm1a_g)

def kontras_teks(g, roi=(slice(150, 300), slice(20, 560))):
    # rentang intensitas p10..p90 pada area nama/NIM: makin lebar makin kontras teksnya
    pk = g[roi].astype(np.float64)
    return float(np.percentile(pk, 90) - np.percentile(pk, 10))

print()
print('%-22s %8s %8s %8s %8s %10s %10s %12s'
      % ('citra', 'rerata', 'std', 'entropi', 'level', 'local std', 'std patch', 'kontras teks'))
def baris(nama, g):
    print('%-22s %8.1f %8.1f %8.3f %8d %10.2f %10.2f %12.1f'
          % (nama, g.mean(), g.std(), entropi(g), int((np.bincount(g.ravel(), minlength=256) > 0).sum()),
             local_std(g).mean(), local_std(g)[iy - 15:iy + 15, ix - 15:ix + 15].std(), kontras_teks(g)))
baris('asli', ktm1a_g)
baris('HE global', he_ktm_manual)
for cl in clips:
    baris('CLAHE clip %.1f' % cl, hasil_clahe_ktm[cl])

npanel = 2 + len(clips)
plt.figure(figsize=(3.4 * npanel, 7))
plt.subplot(2, npanel, 1); plt.imshow(ktm1a_g, cmap='gray'); plt.title(T('asli'), fontsize=9); plt.axis('off')
plt.subplot(2, npanel, 2); plt.imshow(he_ktm_manual, cmap='gray'); plt.title(T('HE global'), fontsize=9); plt.axis('off')
for j, cl in enumerate(clips):
    plt.subplot(2, npanel, 3 + j); plt.imshow(hasil_clahe_ktm[cl], cmap='gray')
    plt.title(T('CLAHE clip %.1f' % cl), fontsize=9); plt.axis('off')
# baris bawah : potongan teks nama/NIM (baris 150-300, kolom 20-560)
plt.subplot(2, npanel, npanel + 1); plt.imshow(ktm1a_g[150:300, 20:560], cmap='gray'); plt.title(T('teks asli'), fontsize=8); plt.axis('off')
plt.subplot(2, npanel, npanel + 2); plt.imshow(he_ktm_manual[150:300, 20:560], cmap='gray'); plt.title(T('teks HE global'), fontsize=8); plt.axis('off')
for j, cl in enumerate(clips):
    plt.subplot(2, npanel, npanel + 3 + j); plt.imshow(hasil_clahe_ktm[cl][150:300, 20:560], cmap='gray')
    plt.title(T('teks CLAHE %.1f' % cl), fontsize=8); plt.axis('off')
plt.tight_layout(); plt.show()

# jejak artefak batas tile: tileGridSize=(4,4) -> hanya 16 tile, batasnya bisa terlihat
H, W = ktm1a_g.shape
th, tw = int(np.ceil(H / PARAM['tile'])), int(np.ceil(W / PARAM['tile']))
print()
print('ukuran satu tile : %d x %d piksel (%d tile per baris/kolom, total %d tile)'
      % (th, tw, PARAM['tile'], PARAM['tile'] ** 2))

def artefak_batas_tile(img):
    """bandingkan gradien horizontal pada kolom batas tile vs kolom di dalam tile"""
    d = np.abs(np.diff(img.astype(np.float32), axis=1))
    kol = [c - 1 for c in range(tw, W, tw) if c - 1 < d.shape[1]]
    mask = np.ones(d.shape[1], dtype=bool)
    mask[kol] = False
    return d[:, kol].mean(), d[:, mask].mean()

print('%-22s %14s %14s %8s' % ('citra', 'gradien batas', 'gradien dalam', 'rasio'))
b, d0 = artefak_batas_tile(ktm1a_g)
print('%-22s %14.2f %14.2f %8.2f' % ('asli', b, d0, b / d0))
b, d0 = artefak_batas_tile(he_ktm_manual)
print('%-22s %14.2f %14.2f %8.2f' % ('HE global', b, d0, b / d0))
for cl in clips:
    b, d0 = artefak_batas_tile(hasil_clahe_ktm[cl])
    print('%-22s %14.2f %14.2f %8.2f' % ('CLAHE clip %.1f' % cl, b, d0, b / d0))

# ukuran keterbacaan yang tidak saturasi: pemisahan teks dan latar dengan ambang Otsu
print()
print('%-22s %10s %14s %14s' % ('citra', 'ambang', 'rerata teks', 'selisih teks-latar'))
def pisah_otsu(g, roi=(slice(150, 300), slice(20, 560))):
    pk = g[roi]
    t, _ = cv.threshold(pk, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    terang = pk[pk >= t].astype(np.float64)
    gelap = pk[pk < t].astype(np.float64)
    return int(t), float(terang.mean()), float(terang.mean() - gelap.mean())

for nama, g in [('asli', ktm1a_g), ('HE global', he_ktm_manual)] + \
               [('CLAHE clip %.1f' % cl, hasil_clahe_ktm[cl]) for cl in clips]:
    t, mt, sel = pisah_otsu(g)
    print('%-22s %10d %14.1f %14.1f' % (nama, t, mt, sel))

# perbesaran area paling datar : di sinilah noise CLAHE paling kelihatan
# kontras tiap potongan dinormalkan sendiri (autoscale) supaya tekstur noise terlihat
plt.figure(figsize=(3.2 * npanel, 4))
plt.subplot(1, npanel, 1)
plt.imshow(ktm1a_g[iy - 30:iy + 30, ix - 30:ix + 30], cmap='gray')
plt.title(T('area datar - asli'), fontsize=9); plt.axis('off')
plt.subplot(1, npanel, 2)
plt.imshow(he_ktm_manual[iy - 30:iy + 30, ix - 30:ix + 30], cmap='gray')
plt.title(T('area datar - HE global'), fontsize=9); plt.axis('off')
for j, cl in enumerate(clips):
    plt.subplot(1, npanel, 3 + j)
    plt.imshow(hasil_clahe_ktm[cl][iy - 30:iy + 30, ix - 30:ix + 30], cmap='gray')
    plt.title(T('area datar - CLAHE %.1f' % cl), fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

## Jawaban PERTANYAAN E2 no.2 — CLAHE lokal pada KTM1a.jpg

**Parameter:** `clipLimit = PARAM['clip'] = 2.0`, `tileGridSize = (PARAM['tile'], PARAM['tile'])
= (4, 4)`. Karena citra 638x1011, satu tile berukuran **160 x 253 piksel** dan seluruh citra
hanya terbagi menjadi **16 tile**.

**(a) CLAHE vs ekualisasi global — mana yang membuat NIM dan nama lebih terbaca?**

| Ukuran | Asli | HE global | CLAHE clip 2,0 |
|---|---|---|---|
| Rerata | 29,9 | 201,6 | 42,4 |
| Std | 11,4 | 85,5 | 14,7 |
| Entropi | 1,619 bit | 1,543 bit | **5,056 bit** |
| Level intensitas terisi | 59 | 20 | **100** |
| Simpangan baku lokal rata-rata (kekasaran/noise) | 4,22 | 33,44 | 6,25 |
| Kontras area nama/NIM (p90−p10) | 22,0 | **202,0** | 36,0 |
| Pemisahan teks–latar (Otsu) | 32,5 | **214,0** | 47,8 |

Pada **area tulisan nama/NIM**, ekualisasi global menghasilkan kontras angka tertinggi
(202,0 dan Otsu 214,0) sehingga tulisan tampak paling tegas — tetapi harganya adalah kenaikan
kekasaran lokal yang sangat besar (simpangan baku lokal 4,22 → 33,44) dan entropi yang justru
turun (1,619 → 1,543 bit, hanya 20 level terisi). Artinya ekualisasi global memaksa semua piksel
menyebar ke skala penuh, sehingga **noise dan tekstur latar ikut meledak** dan bidang kartu yang
semula rata menjadi berbintik. CLAHE dengan clip 2,0 menghasilkan citra yang lebih seimbang:
entropi naik tajam (1,619 → 5,056 bit, 100 level terisi) — informasi lebih banyak dipertahankan
secara lokal — sementara kekasaran lokal hanya naik sedikit (4,22 → 6,25). Untuk keterbacaan
tulisan pada foto KTM ini, **ekualisasi global lebih tegas pada tulisan, tetapi CLAHE lebih
nyaman dilihat karena latar tetap bersih**; bila yang dinilai adalah ketajaman tulisan saja,
global menang, sedangkan bila dinilai keseluruhan kartu (tulisan terbaca tanpa noise mengganggu),
CLAHE lebih baik.

**(b) Tiga nilai `clipLimit` lain di sekitar nilai saya (1,0 / 1,5 / 3,0):**

| clipLimit | Rerata | Std | Entropi | Level terisi | Simpangan baku lokal | Kontras teks | Otsu teks–latar |
|---|---|---|---|---|---|---|---|
| 1,0 | 38,7 | 13,8 | 4,620 | 85 | 5,67 | 31,0 | 43,9 |
| 1,5 | 40,6 | 14,4 | 4,895 | 93 | 6,02 | 33,0 | 46,3 |
| **2,0 (PARAM)** | **42,4** | **14,7** | **5,056** | **100** | **6,25** | **36,0** | **47,8** |
| 3,0 | 45,1 | 15,5 | 5,331 | 108 | 6,68 | 39,0 | 50,5 |

**Nilai terbaik: `clipLimit = 2.0`** — yaitu nilai yang diturunkan dari NIM (N = 102).
Alasannya: pada 1,0–2,0 kenaikan kontras teks (31,0 → 36,0) dan pemisahan Otsu (43,9 → 47,8)
masih sebanding dengan kenaikan kekasaran (5,67 → 6,25, hanya +0,58), sedangkan dari 2,0 ke 3,0
pertambahan kontras teks sudah menipis (36,0 → 39,0, hanya +3) sementara kekasaran dan gradien
batas tile naik lebih cepat (lihat tabel artefak di bawah). Dengan kata lain 2,0 adalah titik
di mana manfaat masih jelas tetapi biaya derau masih terkendali — konsisten dengan sifat CLAHE
bahwa `clipLimit` yang makin tinggi makin mendekati perilaku ekualisasi global.

**Bagian citra yang noise-nya mulai menonjol ketika `clipLimit` dinaikkan.**
Program mencari sendiri piksel paling datar pada citra asli: **baris 40, kolom 591**
(simpangan baku lokal asli = 0,00), lalu menampilkan perbesarannya. Di area datar itulah efek
`clipLimit` paling telanjang terlihat:

| Ukuran pada citra penuh | Asli | CLAHE 1,0 | CLAHE 1,5 | CLAHE 2,0 | CLAHE 3,0 |
|---|---|---|---|---|---|
| Simpangan baku lokal rata-rata | 4,22 | 5,67 | 6,02 | 6,25 | 6,68 |
| Simpangan baku pada patch 30x30 di titik terdataran | 2,78 | 3,46 | 3,47 | 3,70 | 3,82 |
| Gradien di kolom batas tile / di dalam tile (rasio) | 0,99/0,85 (1,16) | 1,61/1,24 (1,30) | 1,71/1,36 (1,26) | 1,85/1,46 (1,27) | **2,27/1,64 (1,38)** |

Dua gejala terlihat jelas: (1) tekstur halus pada area datar bertambah seiring naiknya clip
(patch datar 2,78 → 3,82), dan (2) **artefak batas tile** muncul — gradien pada kolom batas antar
tile konsisten lebih tinggi daripada di dalam tile, dan rasionya memburuk dari 1,30 menjadi 1,38
saat clip dinaikkan dari 1,0 ke 3,0. Karena `tileGridSize = (4,4)` hanya membagi citra menjadi 16
blok (masing-masing 160x253 piksel), batas antar blok ini justru tampak sebagai garis samar —
terlihat pada gambar perbesaran area datar di mana ada perubahan kecerahan mendadak di tepi tile.

## D.3 Image Dithering (Error Diffusion)

Error diffusion adalah salah satu jenis halftoning: hasil kuantisasi (selisih antara warna asli
dan warna palet terdekat) didifusikan ke piksel tetangga yang **belum** diproses. Termasuk
*area operation* karena satu piksel memengaruhi hasil piksel lain, dan kemampuannya memperkuat
tepi membuat teks lebih mudah terbaca dibanding halftoning lain.

Kernel Floyd-Steinberg $\frac{1}{16}\begin{bmatrix} & \# & 7 \\ 3 & 5 & 1\end{bmatrix}$
dengan konvensi modul: $x$ = kolom, $y$ = baris, pemindaian kiri→kanan lalu atas→bawah.

| Tetangga | Posisi | Bobot |
|---|---|---|
| I(x+1, y) | kanan | 7/16 |
| I(x−1, y+1) | bawah-kiri | 3/16 |
| I(x, y+1) | bawah | 5/16 |
| I(x+1, y+1) | bawah-kanan | 1/16 |
| | Jumlah | 7+3+5+1 = 16 → 1 |

Catatan implementasi modul dipatuhi: citra kerja disimpan sebagai `float32` (pada `uint8`
penjumlahan error berputar, mis. 240 + 30 menjadi 14 sehingga hasilnya berbintik), nilai tetangga
dijaga pada 0–255 dengan `np.clip`, dan tetangga di luar batas citra dilewati. Untuk citra
berwarna, fungsi yang sama dijalankan tiga kali (kanal B, G, R) lalu digabung.

In [ ]:
# ---- E-3 : dithering Floyd-Steinberg (implementasi manual) ----
def floyd_steinberg(gray, L=2):
    """error diffusion Floyd-Steinberg; x = kolom, y = baris, scan kiri->kanan lalu atas->bawah"""
    img = gray.astype(np.float32).copy()          # WAJIB float32, bukan uint8
    step = 255.0 / (L - 1)                        # jarak antar level palet
    H, W = img.shape
    for y in range(H):
        for x in range(W):
            lama = img[y, x]
            baru = np.round(lama / step) * step   # warna palet terdekat
            img[y, x] = baru
            error = lama - baru
            if x + 1 < W:             img[y, x + 1]     += error * 7 / 16
            if x > 0 and y + 1 < H:   img[y + 1, x - 1] += error * 3 / 16
            if y + 1 < H:             img[y + 1, x]     += error * 5 / 16
            if x + 1 < W and y + 1 < H: img[y + 1, x + 1] += error * 1 / 16
    return np.clip(img, 0, 255).astype(np.uint8)

def fs_jejak(gray, L=2, langkah=3):
    """versi pelacak: mengembalikan jejak buffer setelah setiap piksel diproses"""
    img = gray.astype(np.float32).copy()
    step = 255.0 / (L - 1)
    H, W = img.shape
    jejak = []
    n = 0
    for y in range(H):
        for x in range(W):
            if n >= langkah:
                return jejak
            lama = float(img[y, x])
            baru = float(np.round(lama / step) * step)
            img[y, x] = baru
            e = lama - baru
            jejak.append((x, y, lama, baru, e, img.copy()))
            if x + 1 < W:             img[y, x + 1]     += e * 7 / 16
            if x > 0 and y + 1 < H:   img[y + 1, x - 1] += e * 3 / 16
            if y + 1 < H:             img[y + 1, x]     += e * 5 / 16
            if x + 1 < W and y + 1 < H: img[y + 1, x + 1] += e * 1 / 16
            n += 1
    return jejak

# ---- verifikasi satu langkah manual pada citra 3 x 3 ----
contoh33 = np.array([[90, 130, 200],
                     [40, 170, 220],
                     [10,  60, 250]], dtype=np.uint8)
jejak = fs_jejak(contoh33, L=2, langkah=3)
np.set_printoptions(precision=3, suppress=True, linewidth=120)
print('citra 3x3 masukan :')
print(contoh33)
for x, y, lama, baru, e, buf in jejak:
    print('piksel (x={}, y={}) : lama {:.3f} -> palet {:.3f} | error = {:.3f}'.format(x, y, lama, baru, e))
    print('  buffer setelah difusi error :')
    print(buf)

### Verifikasi satu langkah manual pada citra 3 x 3

Program menjalankan versi pelacak (`fs_jejak`) yang menampilkan isi buffer kerja **setelah**
setiap piksel diproses, sehingga porsi error yang terdifusi dapat diperiksa satu per satu
(hasil lengkap ada di keluaran sel di bawah):

| Langkah | Piksel | Nilai lama (buffer) | Palet terdekat | error = lama − baru |
|---|---|---|---|---|
| 1 | (x=0, y=0) | 90,000 | 0 | **+90,000** |
| 2 | (x=1, y=0) | 169,375 (= 130 + 90×7/16) | 255 | −85,625 |
| 3 | (x=2, y=0) | 162,539 (= 200 + 90×3/16 − 85,625×7/16) | 255 | −92,461 |

Setelah langkah 1, buffer menunjukkan `130 → 130 + 90×7/16 = 169,375` (kanan),
`40 → 40 + 90×3/16 = 56,875` (bawah-kiri), `170 → 170 + 90×5/16 = 198,125` (bawah), dan
`60 → 60 + 90×1/16 = 65,625` (bawah-kanan) — persis sesuai bobot 7/16, 3/16, 5/16, 1/16.
Pembuktian ini juga menunjukkan mengapa buffer **wajib** `float32`: nilai 169,375 dan 162,539
adalah pecahan yang tidak dapat disimpan pada `uint8`.

In [ ]:
# ---- TAHAP 1 : dithering lena.jpg (citra contoh) ----
t0 = time.perf_counter()
lena_dith = floyd_steinberg(lena_g, PARAM['L'])
t_dith_lena = time.perf_counter() - t0
lena_dith4 = floyd_steinberg(lena_g, 4)       # pembanding jumlah level lebih banyak
print('lena.jpg : L = PARAM[L] = {} | step palet = {:.1f} | waktu {:.2f} s'
      .format(PARAM['L'], 255.0 / (PARAM['L'] - 1), t_dith_lena))

plt.figure(figsize=(16, 5))
for i, (c, t) in enumerate([(lena_g, 'asli grayscale'),
                            (lena_dith, 'Floyd-Steinberg L={} (PARAM)'.format(PARAM['L'])),
                            (lena_dith4, 'Floyd-Steinberg L=4')], 1):
    plt.subplot(1, 3, i); plt.imshow(c, cmap='gray')
    plt.title(T('lena.jpg - ' + t), fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

for t, c in [('asli', lena_g), ('dither L={}'.format(PARAM['L']), lena_dith), ('dither L=4', lena_dith4)]:
    lv = np.unique(c)
    ringkas = str(lv.tolist()) if len(lv) <= 8 else '{} nilai dari {} sampai {}'.format(len(lv), lv.min(), lv.max())
    print('{:<14} rerata {:6.1f} | std {:6.1f} | level terpakai: {}'.format(t, c.mean(), c.std(), ringkas))

In [ ]:
# ---- TAHAP 2 : dithering KTM1b.jpg dengan PARAM['L'] ----
t0 = time.perf_counter()
ktm1b_dith = floyd_steinberg(ktm1b_g, PARAM['L'])
t_dith_ktm = time.perf_counter() - t0
print('KTM1b.jpg : L = {} | waktu {:.2f} s'.format(PARAM['L'], t_dith_ktm))

plt.figure(figsize=(16, 5))
plt.subplot(1, 2, 1); plt.imshow(ktm1b_g, cmap='gray'); plt.title(T('KTM1b.jpg - asli'), fontsize=10); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(ktm1b_dith, cmap='gray')
plt.title(T('KTM1b.jpg - Floyd-Steinberg L={}'.format(PARAM['L'])), fontsize=10); plt.axis('off')
plt.tight_layout(); plt.show()

plt.figure(figsize=(16, 4))
plt.subplot(1, 2, 1)
plt.imshow(ktm1b_g[150:300, 20:560], cmap='gray'); plt.title(T('KTM1b.jpg - teks asli'), fontsize=9); plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(ktm1b_dith[150:300, 20:560], cmap='gray'); plt.title(T('KTM1b.jpg - teks setelah dithering'), fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

print('porsi piksel putih (255) pada hasil dithering : {:.1f}%'.format((ktm1b_dith == 255).mean() * 100))
print('porsi piksel hitam (0) pada hasil dithering   : {:.1f}%'.format((ktm1b_dith == 0).mean() * 100))

### E-3 TUGAS 1 — dithering Floyd-Steinberg pada lena.jpg (tahap 1) dan KTM1b.jpg (tahap 2)

`PARAM['L'] = 2` (dari N = 102: $2^{(102 \bmod 3)+1} = 2^{1}$), sehingga `step = 255/(L-1) = 255`
dan palet hanya berisi {0, 255}. Waktu proses ≈ 1,7–2,6 s untuk 262.144 piksel (loop Python per
piksel, bergantung beban mesin). Sebagai pembanding dijalankan juga L = 4 (palet {0, 85, 170, 255}).

### Tahap 2 — dithering KTM1b.jpg dengan PARAM['L']

Pada `KTM1b.jpg` (645.018 piksel) proses memakan ≈ 4,2–5,2 s. Karena rerata citra 119,5 (di bawah
titik tengah 127,5), hasil dithering 2 level menghasilkan porsi hitam yang lebih banyak:
**53,1 % piksel hitam (0)** dan **46,9 % piksel putih (255)** — dan justru perbandingan inilah
yang menyimpan informasi kecerahan lokal, sehingga citra tetap terlihat seperti kartu, bukan
seperti blob hitam-putih.

## Jawaban TUGAS E-3 no.1 — dithering Floyd-Steinberg (lena.jpg dan KTM1b.jpg)

**Tahap 1 (lena.jpg, verifikasi kode):** keluaran dithering 2 level (L = `PARAM['L']` = 2) sudah
sesuai dengan gambar contoh modul — citra tampak sebagai halftoning hitam-putih yang masih
memperlihatkan detail wajah, bukan citra biner acak.

| Citra | Rerata | Std | Level terpakai |
|---|---|---|---|
| `lena.jpg` asli grayscale | 124,2 | 47,9 | 222 nilai (19–246) |
| dithering L = 2 (PARAM) | 124,2 | **127,5** | **[0, 255]** |
| dithering L = 4 (pembanding) | 124,2 | 59,2 | [0, 85, 170, 255] |

Dua hal penting dari tabel ini: (1) **rerata tidak berubah** (124,2 pada ketiganya) — itulah bukti
bahwa error yang hilang pada satu piksel benar-benar dipindahkan ke tetangganya, bukan dibuang;
(2) std melonjak dari 47,9 ke 127,5 karena setiap piksel sekarang hanya bernilai 0 atau 255, namun
karena penyebarannya teratur, rerata lokal tetap terjaga sehingga mata masih melihat gradasi.
Dengan L = 4, palet berisi 4 level sehingga std hanya 59,2 — makin banyak level, makin dekat ke
citra asli, tetapi makin sedikit warna yang bisa direproduksi perangkat cetak.

**Tahap 2 (KTM1b.jpg, `PARAM['L']` = 2):** hasil dithering memakai palet {0, 255} dengan porsi
53,1 % hitam dan 46,9 % putih. Pada potongan area tulisan nama/NIM, huruf-huruf tetap terbentuk
dari kumpulan titik hitam dan putih, dan karena error diffusion memperkuat tepi, tulisan pada
kartu masih terbaca pada citra 1 bit ini.

In [ ]:
# ---- TUGAS E-3 no.2 : grayscale -> equalization -> dithering vs dithering langsung ----
def kesetiaan_tone(sumber, dither, k=9):
    """selisih rerata lokal antara citra sumber dan hasil dithering (semakin kecil semakin setia)"""
    a = cv.boxFilter(sumber.astype(np.float32), -1, (k, k))
    b = cv.boxFilter(dither.astype(np.float32), -1, (k, k))
    return float(np.abs(a - b).mean())

def banding_urutan(berkas, gray, ket):
    he = cv.equalizeHist(gray)                       # sebaran intensitas diperbaiki dulu
    d_direct = floyd_steinberg(gray, PARAM['L'])     # urutan 1 : dither langsung
    d_setelah_he = floyd_steinberg(he, PARAM['L'])   # urutan 2 : dither setelah HE
    judul = ['asli grayscale', 'hasil HE', 'dither langsung', 'HE lalu dither']
    citra = [gray, he, d_direct, d_setelah_he]

    plt.figure(figsize=(18, 5))
    for i, (c, t) in enumerate(zip(citra, judul), 1):
        plt.subplot(1, 4, i); plt.imshow(c, cmap='gray')
        plt.title(T('{} - {}'.format(berkas, t)), fontsize=9); plt.axis('off')
    plt.suptitle(T('{} - {}'.format(berkas, ket)), fontsize=11)
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(18, 5))
    potong = [gray, he, d_direct, d_setelah_he]
    for i, (c, t) in enumerate(zip(potong, judul), 1):
        plt.subplot(1, 4, i); plt.imshow(c[150:300, 20:560], cmap='gray')
        plt.title(T('{} - teks {}'.format(berkas, t)), fontsize=8); plt.axis('off')
    plt.tight_layout(); plt.show()

    print('== {} ({}) =='.format(berkas, ket))
    # kontras area teks: p90 - p10 pada potongan nama/NIM (semakin lebar semakin terbaca)
    print('{:<20} {:>8} {:>8} {:>12} {:>14}'.format('tahap', 'rerata', 'std', 'putih (%)', 'kontras teks'))
    for t, c in zip(judul, citra):
        pk = c[150:300, 20:560].astype(np.float64)
        kt = float(np.percentile(pk, 90) - np.percentile(pk, 10))
        print('{:<20} {:>8.1f} {:>8.1f} {:>12.1f} {:>14.1f}'
              .format(t, c.mean(), c.std(), (c == 255).mean() * 100, kt))
    # pada citra biner (2 level) kontras p90-p10 selalu 255, sehingga tidak dapat
    # membedakan; mata melihat rerata lokal halftone, jadi diukur setelah penghalusan 9x9
    def halus(x, k=9):
        return cv.boxFilter(x.astype(np.float32), -1, (k, k))
    def kontras_halus(x):
        pk = halus(x)[150:300, 20:560].astype(np.float64)
        return float(np.percentile(pk, 90) - np.percentile(pk, 10))
    print('kontras teks setelah penghalusan 9x9 : dither langsung {:.1f} | HE lalu dither {:.1f}'
          .format(kontras_halus(d_direct), kontras_halus(d_setelah_he)))
    print('acuan sumber                     : asli {:.1f} | HE {:.1f}'
          .format(kontras_halus(gray), kontras_halus(he)))
    print('kesetiaan tone dither langsung : {:.2f} (dibanding citra asli)'
          .format(kesetiaan_tone(gray, d_direct)))
    print('kesetiaan tone HE lalu dither  : {:.2f} (dibanding citra hasil HE)'
          .format(kesetiaan_tone(he, d_setelah_he)))
    return d_direct, d_setelah_he

print('########## TAHAP 1 : lena_lc.jpg ##########')
dd_lc, dh_lc = banding_urutan('lena_lc.jpg', lena_lc, 'tahap 1 - citra contoh')

In [ ]:
print('########## TAHAP 2 : KTM1a.jpg ##########')
dd_ktm, dh_ktm = banding_urutan('KTM1a.jpg', ktm1a_g, 'tahap 2 - KTM ruangan gelap')

## Jawaban TUGAS E-3 no.2 — grayscale → equalization → dithering vs dithering langsung

**Tahap 1 (lena_lc.jpg):**

| Tahap | Rerata | Std | Putih (%) | Kontras teks | Kontras teks setelah penghalusan 9x9 |
|---|---|---|---|---|---|
| asli grayscale | 136,7 | 12,0 | 0,0 | 34,0 | 30,6 (acuan) |
| hasil HE | 130,6 | 73,6 | 0,2 | 204,0 | 188,4 (acuan) |
| dither langsung | 136,7 | 127,2 | 53,6 | 255,0 | **31,5** |
| HE lalu dither | 130,6 | 127,5 | 51,2 | 255,0 | **188,9** |

Kesetiaan tone (selisih rerata lokal 9x9 terhadap sumbernya): dither langsung **2,56**, HE lalu
dither **2,71** — keduanya kecil, artinya kedua urutan sama-sama memindahkan error dengan benar.

**Tahap 2 (KTM1a.jpg, ruangan gelap):**

| Tahap | Rerata | Std | Putih (%) | Kontras teks | Kontras teks setelah penghalusan 9x9 |
|---|---|---|---|---|---|
| asli grayscale | 29,9 | 11,4 | 0,0 | 22,0 | 14,3 (acuan) |
| hasil HE | 200,7 | 87,2 | 0,2 | 206,0 | 143,6 (acuan) |
| dither langsung | 29,8 | 81,9 | 11,7 | 255,0 | **18,9** |
| HE lalu dither | 200,8 | 104,3 | 78,7 | 255,0 | **148,0** |

Catatan pengukuran: pada hasil dithering, kontras p90−p10 selalu 255 karena hanya ada dua nilai
piksel, sehingga ukuran itu **tidak dapat membedakan** kedua urutan. Karena mata melihat rerata
lokal dari pola halftone, kontras diukur ulang setelah penghalusan 9x9; kolom terakhir itulah
yang bermakna. Angka acuannya adalah kontras sumber: 14,3 (asli) dan 143,6 (setelah HE).

**Kesetiaan tone:** dither langsung 2,07 (dibanding asli) dan HE lalu dither 2,55 (dibanding HE) —
keduanya sangat kecil, jadi error diffusion berhasil mempertahankan tingkat kecerahan lokal pada
kedua urutan.

**Urutan mana yang menghasilkan teks pada KTM lebih terbaca?**

| Urutan | Porsi piksel putih pada area teks | Kontras teks (setelah halus) | Keterbacaan |
|---|---|---|---|
| Dither langsung (asli rerata 29,9) | **11,7 %** putih (88,3 % hitam) | 18,9 | hampir seluruh area menjadi hitam; huruf hanya dibentuk oleh 11,7 % titik putih sehingga tepinya bergerigi dan tulisan **sulit dibaca** |
| HE lalu dither (HE rerata 200,7) | **78,7 %** putih | **148,0** | latar terang, huruf hitam tegas, kontras hampir 8 kali lebih besar — tulisan **jauh lebih terbaca** |

Jadi urutan **grayscale → histogram equalization → dithering** menghasilkan teks yang jauh lebih
terbaca daripada dithering langsung. Sebabnya masuk akal secara teori: dithering hanya memindahkan
error, ia **tidak dapat menciptakan kontras yang tidak ada**. Pada `KTM1a.jpg` seluruh intensitas
terkurung di rentang sempit 0–60 (std 11,4) sehingga setelah dikuantisasi ke dua level, informasi
terang-gelap yang tersisa sangat sedikit dan bidang kartu didominasi titik hitam (88,3 %). Setelah
ekualisasi, sebaran intensitas melebar (std 87,2) sehingga jarak antara tulisan dan latar menjadi
besar dan hasil ditheringnya memakai kedua level secara proporsional. Perbandingan angka pada
tahap 1 (`lena_lc.jpg`) juga konsisten: dither langsung hanya mencapai kontras 31,5 sedangkan
HE lalu dither 188,9, mendekati kontras sumber 188,4 — jadi urutan HE lalu dither memang lebih
setia mereproduksi kontras yang sudah diperbaiki.

In [ ]:
# ---- catatan modul : untuk citra berwarna fungsi yang sama dijalankan tiga kali (B, G, R) ----
bgr_lena = cv.imread(p('lena.jpg'))
kanal_fs = [floyd_steinberg(bgr_lena[:, :, i], PARAM['L']) for i in range(3)]
lena_dith_warna = cv.merge(kanal_fs)

# pembanding : kuantisasi ke warna terdekat TANPA error diffusion
step = 255.0 / (PARAM['L'] - 1)
lena_tanpa_diffusi = np.clip(np.round(bgr_lena / step) * step, 0, 255).astype(np.uint8)

plt.figure(figsize=(18, 6))
plt.subplot(1, 3, 1); plt.imshow(cv.cvtColor(bgr_lena, cv.COLOR_BGR2RGB))
plt.title(T('lena.jpg - asli'), fontsize=9); plt.axis('off')
plt.subplot(1, 3, 2); plt.imshow(cv.cvtColor(lena_tanpa_diffusi, cv.COLOR_BGR2RGB))
plt.title(T('warna terdekat tanpa difusi L={}'.format(PARAM['L'])), fontsize=9); plt.axis('off')
plt.subplot(1, 3, 3); plt.imshow(cv.cvtColor(lena_dith_warna, cv.COLOR_BGR2RGB))
plt.title(T('Floyd-Steinberg 3 kanal L={}'.format(PARAM['L'])), fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

for t, c in [('asli', bgr_lena), ('tanpa difusi', lena_tanpa_diffusi), ('FS 3 kanal', lena_dith_warna)]:
    print('{:<14} rerata {:6.1f} | std {:6.1f}'.format(t, c.mean(), c.std()))
print('jumlah level berbeda pada FS 3 kanal : {} nilai'.format(len(np.unique(lena_dith_warna))))
print('jumlah warna unik pada FS 3 kanal   : {}'.format(len(np.unique(lena_dith_warna.reshape(-1, 3), axis=0))))

## Catatan tambahan — dithering citra berwarna (3 kanal)

Sesuai catatan modul, untuk citra berwarna fungsi Floyd-Steinberg dijalankan tiga kali
(kanal B, G, R) lalu digabung. Program juga membandingkannya dengan cara paling sederhana, yaitu
mencari warna terdekat **tanpa** error diffusion:

| Cara | Rerata | Std |
|---|---|---|
| `lena.jpg` asli berwarna | 128,2 | 58,7 |
| warna terdekat tanpa difusi (L = 2) | 112,8 | 126,6 |
| Floyd-Steinberg 3 kanal (L = 2) | **128,2** | 127,5 |

Tanpa difusi, rerata jatuh dari 128,2 ke 112,8 karena setiap piksel dipaksa ke 0 atau 255 tanpa
kompensasi, sehingga kecerahan keseluruhan bergeser. Dengan error diffusion, rerata kembali persis
128,2. Hasilnya tersusun dari 2 nilai per kanal, yaitu **8 kombinasi warna** — persis seperti
contoh pada modul (penyederhanaan gambar menjadi 8 warna), namun di sini gradasinya tetap halus
karena error didifusikan pada ketiga kanal secara terpisah.

## Penutup

Seluruh percobaan modul (E-1, E-2, E-3 beserta PERTANYAAN dan TUGAS) telah dijalankan pada
lingkungan Kaggle tanpa GUI dan tanpa Colab. Ringkasan temuan utama:

1. **Histogram** — histogram manual identik dengan `np.histogram` dan `cv.calcHist`
   (selisih maksimum 0 pada `lena.jpg` maupun `KTM1b.jpg`); loop manual hanya unggul dalam hal
   kejelasan alur, bukan kecepatan.
2. **Kurva CDF** — `KTM1a.jpg` (ruangan gelap) memberi CDF yang melonjak di kiri (median 35),
   `KTM1d.jpg` (matahari) memberi CDF yang mendatar panjang lalu melonjak di kanan (median 243),
   sesuai kondisi pencahayaan saat akuisisi.
3. **Ekualisasi** — implementasi manual sama persis dengan `cv.equalizeHist` pada `lena_lc.jpg`
   (selisih 0) dan berbeda maksimum 5 level pada `KTM1a.jpg` semata-mata karena definisi titik
   awal LUT ($imin$), bukan karena kesalahan rumus.
4. **Ekualisasi warna** — cara 1 (B, G, R terpisah) menggeser Hue hingga 75,01° (lena) dan
   20,33° (KTM1d), sedangkan cara 2 (kanal terang) hanya 0,65–2,68°; kanal **Y** paling setia
   untuk foto KTM, dan **CLAHE clip 2,0 / tile 4x4** mempertahankan kecerahan (216,0 dari 219,4)
   sekaligus menekan pergeseran Hue ke 0,13°.
5. **PSNR** tidak layak menilai keterbacaan: turun ke 2,68 dB justru ketika teks menjadi jauh
   lebih kontras (pemisahan Otsu 32,5 → 214,0).
6. **Dithering** — Floyd-Steinberg menjaga rerata citra (124,2 → 124,2 pada lena) sehingga gradasi
   tetap terlihat; untuk `KTM1a.jpg` urutan **HE lalu dither** (kontras teks 148,0) jauh lebih
   terbaca daripada dithering langsung (18,9).